# RNA-seq Analysis — Module I Final Project

**Course:** Bioinformatics Methods for Transcriptomics  
**Institution:** Johns Hopkins University

## 01. Project Overview

This project investigates changes in gene expression during mouse hippocampus development using RNA-seq data.

The dataset contains six biological samples: three from 1-month-old mice and three from 4-month-old mice.

The analysis uses a 3.4 Mb genomic region from mouse chromosome 7 (GRCm39) and the corresponding GENCODE vM30 annotation.

### Workflow

1. Prepare and validate the input files.
2. Build the STAR genome index.
3. Align RNA-seq reads using STAR.
4. Examine alignment statistics using STAR and SAMtools.
5. Assemble transcripts using PsiCLASS.
6. Annotate transcripts using GENCODE.
7. Analyze differential expression using Cuffdiff2.
8. Generate gene-level counts and analyze them using DESeq2.
9. Visualize and interpret the results.

## 02. Environment Setup & Dependencies

Import the Python libraries required for file management, data processing, and execution of external tools.

Check the computational environment and identify the bioinformatics software used in the analysis.

#### Python: Imports

In [3]:
from pathlib import Path                        # Manage file and directory paths
from collections import defaultdict, Counter    # Group data and count occurrences
import importlib.util                           # Check Python package availability
import os                                       # Access environment variables
import re                                       # Process text patterns
import shutil                                   # Locate executables
import subprocess                               # Run external commands
import sys                                      # Access Python information
import tempfile                                 # Create temporary files and directories

# Check the current Python environment
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Conda:", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))

Python: 3.11.6
Executable: /home/ianca/miniconda3/envs/transcriptomics/bin/python
Conda: transcriptomics


#### Python: Python dependencies

In [4]:
# Install missing Python packages
PACKAGES = ["numpy", "pandas", "matplotlib", "seaborn"]

INSTALL_MISSING = False

for package in PACKAGES:
    if importlib.util.find_spec(package) is not None:
        print(f"[OK] {package}")
    elif INSTALL_MISSING:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", package],
            check=True
        )
        print(f"[INSTALLED] {package}")
    else:
        print(f"[MISSING] {package}")

import pandas as pd
import numpy as np

[OK] numpy
[OK] pandas
[OK] matplotlib
[OK] seaborn


#### Python: Bioinformatics software

In [5]:
# Check command-line tools
TOOLS = {
    "STAR": "STAR",
    "SAMtools": "samtools",
    "PsiCLASS": "psiclass",
    "Cuffdiff2": "cuffdiff",
    "R": "Rscript"
}

for name, executable in TOOLS.items():
    path = shutil.which(executable)
    print(f"{name}: {path if path else 'Not found in current PATH'}")

# PsiCLASS is installed in a separate Conda environment
PSICLASS_EXE = Path.home() / "miniconda3/envs/psiclass/bin/psiclass"

print("\nPsiCLASS dedicated environment:")
print("[OK]" if PSICLASS_EXE.is_file() else "[MISSING]", PSICLASS_EXE)

# Check DESeq2
rscript = shutil.which("Rscript")

if rscript:
    result = subprocess.run(
        [
            rscript, "-e",
            'if (requireNamespace("DESeq2", quietly=TRUE)) '
            'cat(as.character(packageVersion("DESeq2"))) '
            'else cat("Not installed")'
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print("DESeq2:", result.stdout.strip() or "Not detected")

STAR: /usr/bin/STAR
SAMtools: /usr/bin/samtools
PsiCLASS: Not found in current PATH
Cuffdiff2: /usr/bin/cuffdiff
R: /home/ianca/miniconda3/envs/transcriptomics/bin/Rscript

PsiCLASS dedicated environment:
[OK] /home/ianca/miniconda3/envs/psiclass/bin/psiclass
DESeq2: 1.50.2


## 03. Project Configuration

Define the project directories, reference files, sample identifiers, and analysis parameters in one place.

These settings are reused throughout the workflow to avoid repeated paths and inconsistent configurations.

In [6]:
# Project directories
REPO_DIR = Path("/mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project")
DATA_DIR = Path("/mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project")

# Input files
RAW_DIR = REPO_DIR / "data_raw"
FASTA = RAW_DIR / "region.fa"
GENCODE_GTF = RAW_DIR / "gencode.vM30.region.gtf"
METADATA_FILE = RAW_DIR / "metadata.tsv"

# Scripts and results
SCRIPTS_DIR = REPO_DIR / "scripts"
RESULTS_DIR = DATA_DIR / "results"

# STAR
STAR_DIR = RESULTS_DIR / "STAR"
STAR_INDEX_DIR = STAR_DIR / "star_index"
STAR_TMP_DIR = Path.home() / "star_tmp_final_project"

# PsiCLASS
PSICLASS_DIR = RESULTS_DIR / "PsiCLASS"
BAMLIST = PSICLASS_DIR / "bamlist.psiclass"
PSICLASS_VOTE = PSICLASS_DIR / "psiclass_vote.gtf"
ANNOTATION_DIR = PSICLASS_DIR / "WithGeneNames"

# Downstream analysis
CUFFDIFF_DIR = RESULTS_DIR / "Cuffdiff"
SORTED_DIR = RESULTS_DIR / "Sorted"
DESEQ2_DIR = RESULTS_DIR / "DESeq2"
DESEQ2_RESULTS_DIR = DESEQ2_DIR / "Results"

# Samples
SAMPLES_1MO = ["1mo_Rep1", "1mo_Rep2", "1mo_Rep3"]
SAMPLES_4MO = ["4mo_Rep1", "4mo_Rep2", "4mo_Rep3"]
SAMPLES = SAMPLES_1MO + SAMPLES_4MO

# Parameters
THREADS = 2
SA_INDEX_NBASES = 9
FASTQ_EXTENSION = ".fastq.gz"

# Execution controls
RUN_STAR_INDEX = False
RUN_STAR_ALIGNMENT = False
RUN_PSICLASS = False
RUN_CUFFDIFF = False
RUN_READ_COUNTING = False
RUN_DESEQ2 = False

print("Project configuration loaded.")
print("Samples:", len(SAMPLES))
print("Raw data:", RAW_DIR)
print("Results:", RESULTS_DIR)

Project configuration loaded.
Samples: 6
Raw data: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw
Results: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results


In [7]:
# Check current file paths
print("FASTA:", FASTA)
print("FASTA exists:", FASTA.is_file())

print("\nGENCODE:", GENCODE_GTF)
print("GENCODE exists:", GENCODE_GTF.is_file())

print("\nMetadata:", METADATA_FILE)
print("Metadata exists:", METADATA_FILE.is_file())

FASTA: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/region.fa
FASTA exists: True

GENCODE: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/gencode.vM30.region.gtf
GENCODE exists: True

Metadata: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/metadata.tsv
Metadata exists: True


## 04. Input Validation

Verify that the six FASTQ files, reference genome, gene annotation, and sample metadata are available.

Inspect the metadata and confirm that the sample identifiers match the sequencing files.

This step does not modify the input data.

In [8]:
# Check reference and metadata files
for file in [FASTA, GENCODE_GTF, METADATA_FILE]:
    if not file.is_file():
        raise FileNotFoundError(file)
    print(f"[OK] {file.name}")

# Check FASTQ files
for sample in SAMPLES:
    fastq = RAW_DIR / f"{sample}{FASTQ_EXTENSION}"

    if not fastq.is_file():
        raise FileNotFoundError(fastq)

    print(f"[OK] {sample}")

# Read sample metadata
metadata = pd.read_csv(METADATA_FILE, sep=r"\s+")

print("\nSample metadata:")
display(metadata)

# Verify sample identifiers
if set(metadata["Run"]) != set(SAMPLES):
    raise ValueError("Metadata and FASTQ samples do not match.")

# Verify experimental groups
expected_groups = {"1_Month": 3, "4_Month": 3}
observed_groups = metadata["Age"].value_counts().to_dict()

if observed_groups != expected_groups:
    raise ValueError("Unexpected experimental groups.")

print("\nAll input files and sample groups verified successfully!")

[OK] region.fa
[OK] gencode.vM30.region.gtf
[OK] metadata.tsv
[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

Sample metadata:


,Run,Age
0,1mo_Rep1,1_Month
1,1mo_Rep2,1_Month
2,1mo_Rep3,1_Month
3,4mo_Rep1,4_Month
4,4mo_Rep2,4_Month
5,4mo_Rep3,4_Month



All input files and sample groups verified successfully!


In [9]:
# Count genes in the provided GENCODE annotation
gene_ids = set()
gene_features = 0

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "gene":
            continue

        gene_features += 1

        match = re.search(r'gene_id "([^"]+)"', fields[8])
        if match:
            gene_ids.add(match.group(1))

print("GENCODE gene features:", gene_features)
print("Unique GENCODE genes:", len(gene_ids))

GENCODE gene features: 166
Unique GENCODE genes: 166


## 05. STAR Genome Indexing

Generate a STAR genome index from the provided `region.fa` reference sequence.

The reference contains approximately 3.4 million bases. The `genomeSAindexNbases` parameter is set to 9 to accommodate the reduced reference size.

An existing index is reused. Index generation is performed only when explicitly enabled.

In [10]:
# Locate existing STAR genome indices
search_dirs = [
    DATA_DIR,
    DATA_DIR.parent,
    REPO_DIR
]

for directory in search_dirs:
    print(f"\nSearching: {directory}")

    if directory.is_dir():
        for file in directory.rglob("genomeParameters.txt"):
            print("[FOUND]", file.parent)


Searching: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project
[FOUND] /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index
[FOUND] /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index_sparse1

Searching: /mnt/d/Bioinformatics/transcriptomics-data/Module_I
[FOUND] /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index
[FOUND] /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index_sparse1
[FOUND] /mnt/d/Bioinformatics/transcriptomics-data/Module_I/reference/star_index

Searching: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project


In [11]:
# Required STAR index files
INDEX_FILES = ["Genome", "SA", "SAindex", "genomeParameters.txt"]

index_ready = all(
    (STAR_INDEX_DIR / name).is_file()
    for name in INDEX_FILES
)

if index_ready:
    print("[OK] STAR index already exists.")

elif RUN_STAR_INDEX:
    if STAR_INDEX_DIR.exists() and any(STAR_INDEX_DIR.iterdir()):
        raise FileExistsError("Incomplete STAR index. Inspect it before rerunning.")

    STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--runMode", "genomeGenerate",
        "--genomeDir", str(STAR_INDEX_DIR),
        "--genomeFastaFiles", str(FASTA),
        "--genomeSAsparseD", "16",
        "--genomeSAindexNbases", str(SA_INDEX_NBASES)
    ]

    subprocess.run(command, check=True)

else:
    print("[MISSING] STAR index. Set RUN_STAR_INDEX = True to generate it.")

# Verify available index files
for name in INDEX_FILES:
    path = STAR_INDEX_DIR / name
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

[OK] STAR index already exists.
Genome: FOUND
SA: FOUND
SAindex: FOUND
genomeParameters.txt: FOUND


## 06. STAR Read Alignment

Align the six compressed FASTQ files to the indexed reference genome using STAR.

Each sample has a separate output directory containing its coordinate-sorted BAM file and alignment logs.

Temporary files are stored in the Linux filesystem to avoid FIFO compatibility issues with Windows-mounted drives.

Existing alignments are checked before execution and are not overwritten automatically.

In [12]:
# Locate existing STAR outputs
def star_files(sample):
    folder = STAR_DIR / sample

    nested_bam = folder / "Aligned.sortedByCoord.out.bam"
    nested_log = folder / "Log.final.out"

    flat_bam = STAR_DIR / f"{sample}_Aligned.sortedByCoord.out.bam"
    flat_log = STAR_DIR / f"{sample}_Log.final.out"

    if nested_bam.is_file():
        return nested_bam, nested_log

    if flat_bam.is_file():
        return flat_bam, flat_log

    return nested_bam, nested_log


# Check and optionally align samples
for sample in SAMPLES:
    bam, log = star_files(sample)

    print(f"\nSample: {sample}")

    if bam.is_file():
        subprocess.run(
            ["samtools", "quickcheck", "-v", str(bam)],
            check=True
        )
        print("[OK] BAM:", bam)
        print("[OK] Log" if log.is_file() else "[MISSING] Log.final.out")
        continue

    print("[MISSING] BAM")

    if not RUN_STAR_ALIGNMENT:
        continue

    sample_dir = STAR_DIR / sample

    if sample_dir.exists() and any(sample_dir.iterdir()):
        raise FileExistsError(f"Existing files require inspection: {sample_dir}")

    sample_dir.mkdir(parents=True, exist_ok=True)
    STAR_TMP_DIR.mkdir(parents=True, exist_ok=True)

    fastq = RAW_DIR / f"{sample}{FASTQ_EXTENSION}"
    temp_dir = STAR_TMP_DIR / f"{sample}_STARtmp"

    if temp_dir.exists():
        raise FileExistsError(f"Existing STAR temporary directory: {temp_dir}")

    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(STAR_INDEX_DIR),
        "--readFilesIn", str(fastq),
        "--readFilesCommand", "zcat",
        "--outSAMtype", "BAM", "SortedByCoordinate",
        "--outSAMstrandField", "intronMotif",
        "--outTmpDir", str(temp_dir),
        "--outFileNamePrefix", str(sample_dir) + "/"
    ]

    subprocess.run(command, check=True)

print("\nSTAR alignment check completed.")


Sample: 1mo_Rep1
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/1mo_Rep1/Aligned.sortedByCoord.out.bam
[OK] Log

Sample: 1mo_Rep2
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/1mo_Rep2/Aligned.sortedByCoord.out.bam
[OK] Log

Sample: 1mo_Rep3
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/1mo_Rep3/Aligned.sortedByCoord.out.bam
[OK] Log

Sample: 4mo_Rep1
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/4mo_Rep1/Aligned.sortedByCoord.out.bam
[OK] Log

Sample: 4mo_Rep2
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/4mo_Rep2/Aligned.sortedByCoord.out.bam
[OK] Log

Sample: 4mo_Rep3
[OK] BAM: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/4mo_Rep3/Aligned.sortedByCoord.out.bam
[OK] Log

STAR alignment check completed.


In [1]:
# Inspect existing STAR alignment files
print("STAR directory:", STAR_DIR)

# Find BAM files and alignment logs
bam_files = sorted(STAR_DIR.rglob("*.bam"))
log_files = sorted(STAR_DIR.rglob("Log.final.out"))
log_files += sorted(STAR_DIR.glob("*_Log.final.out"))

print("\nBAM files:")
for bam in bam_files:
    print(bam.relative_to(STAR_DIR))

print("\nSTAR alignment logs:")
for log in log_files:
    print(log.relative_to(STAR_DIR))

print(f"\nTotal BAM files: {len(bam_files)}")
print(f"Total alignment logs: {len(log_files)}")

NameError: name 'STAR_DIR' is not defined

In [14]:
# Build a comparison STAR index using default suffix array sparsity
TEST_INDEX_DIR = STAR_DIR / "star_index_sparse1"

required_files = [
    TEST_INDEX_DIR / "Genome",
    TEST_INDEX_DIR / "SA",
    TEST_INDEX_DIR / "SAindex"
]

if all(p.is_file() and p.stat().st_size > 0 for p in required_files):
    print("[OK] Comparison STAR index already exists.")

elif TEST_INDEX_DIR.exists() and any(TEST_INDEX_DIR.iterdir()):
    raise FileExistsError(
        "Partial comparison index found. Inspect before rerunning."
    )

else:
    TEST_INDEX_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "STAR",
        "--runMode", "genomeGenerate",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(TEST_INDEX_DIR),
        "--genomeFastaFiles", str(RAW_DIR / "region.fa"),
        "--genomeSAindexNbases", "9",
        "--genomeSAsparseD", "1"
    ]

    subprocess.run(command, check=True)

print("Comparison index:", TEST_INDEX_DIR)

[OK] Comparison STAR index already exists.
Comparison index: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index_sparse1


## 07. Alignment Quality Assessment

Inspect the BAM files and STAR alignment reports for all six samples.

The summary includes input reads, uniquely mapped reads, reads mapped to multiple loci, and reads that were too short to map.

BAM alignment records and the number of mapped reads are distinct measurements and should not be interpreted interchangeably.

In [15]:
# Validate STAR alignment files
for sample in SAMPLES:
    bam, log = star_files(sample)

    if not bam.is_file():
        raise FileNotFoundError(f"Missing BAM: {bam}")

    if not log.is_file():
        raise FileNotFoundError(f"Missing STAR log: {log}")

    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True
    )

    print(f"[OK] {sample}")

print("\nAll six BAM files passed basic validation.")

[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

All six BAM files passed basic validation.


In [16]:
# Collect alignment statistics for all samples
alignment_summary = []

for sample in SAMPLES:
    bam, log = star_files(sample)

    if not bam.is_file() or not log.is_file():
        print(f"[MISSING] Files for {sample}")
        continue

    # Count BAM alignment records
    result = subprocess.run(
        ["samtools", "view", "-c", str(bam)],
        capture_output=True,
        text=True,
        check=True
    )

    alignment_records = int(result.stdout.strip())

    # Read STAR summary
    stats = {}

    with log.open(encoding="utf-8", errors="replace") as file:
        for line in file:
            if "|" in line:
                key, value = line.split("|", 1)
                stats[key.strip()] = value.strip()

    def read_stat(key):
        value = stats.get(key, "0").replace(",", "")
        return int(value)

    alignment_summary.append({
        "Sample": sample,
        "Input reads": read_stat("Number of input reads"),
        "Unique reads": read_stat("Uniquely mapped reads number"),
        "Multiple loci": read_stat("Number of reads mapped to multiple loci"),
        "Too short": read_stat("Number of reads unmapped: too short"),
        "BAM records": alignment_records
    })

alignment_df = pd.DataFrame(alignment_summary)

display(alignment_df)

,Sample,Input reads,Unique reads,Multiple loci,Too short,BAM records
0,1mo_Rep1,32447,30970,1022,45,33304
1,1mo_Rep2,40653,38976,1191,40,41670
2,1mo_Rep3,40042,38325,1241,47,41141
3,4mo_Rep1,35269,33663,1177,34,36334
4,4mo_Rep2,37649,36014,1206,47,38706
5,4mo_Rep3,41626,39726,1361,43,42851


In [17]:
# Inspect STAR alignment settings and reference configuration
sample = "1mo_Rep2"
bam, log = star_files(sample)

star_log = bam.parent / "Log.out"
genome_parameters = STAR_INDEX_DIR / "genomeParameters.txt"

# Inspect STAR execution parameters
print("STAR execution parameters:")
if star_log.is_file():
    for line in star_log.read_text(errors="replace").splitlines():
        if any(term in line for term in [
            "STAR version",
            "genomeDir",
            "readFilesIn",
            "outSAMstrandField",
            "outFilter",
            "alignIntron"
        ]):
            print(line)
else:
    print("Log.out not found.")

# Inspect genome index configuration
print("\nSTAR genome index parameters:")
if genome_parameters.is_file():
    for line in genome_parameters.read_text().splitlines():
        if any(term in line for term in [
            "genomeFastaFiles",
            "genomeSAindexNbases",
            "genomeSAsparseD"
        ]):
            print(line)

# Compare alignment statistics with course feedback
expected = {
    "Uniquely mapped reads number": 40076,
    "Number of reads mapped to multiple loci": 570,
    "Number of reads unmapped: too short": 1
}

observed = {}

for line in log.read_text().splitlines():
    if "|" in line:
        key, value = line.split("|", 1)
        observed[key.strip()] = value.strip()

print("\nAlignment comparison — 1mo_Rep2:")

for key, expected_value in expected.items():
    actual = int(observed[key].replace(",", ""))

    print(
        f"{key}: "
        f"observed={actual}, "
        f"expected={expected_value}, "
        f"difference={actual - expected_value}"
    )

STAR execution parameters:
STAR version=2.7.11b
/usr/lib/rna-star/bin/STAR-avx2 --runThreadN 2 --genomeDir /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index --readFilesIn /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/1mo_Rep2.fastq.gz --readFilesCommand zcat --outSAMtype BAM SortedByCoordinate --outSAMstrandField intronMotif --outTmpDir /home/ianca/star_tmp_final_project/1mo_Rep2_STARtmp --outFileNamePrefix /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/1mo_Rep2/
genomeDir                     /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index     ~RE-DEFINED
readFilesIn                   /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/1mo_Rep2.fastq.gz        ~RE-DEFINED
outSAMstrandField             intronMotif     ~RE-DEFINED
genomeDir                    

In [18]:
# Test STAR alignment using default suffix array sparsity
TEST_SAMPLE = "1mo_Rep2"
TEST_ALIGN_DIR = STAR_DIR / "test_sparse1" / TEST_SAMPLE
TEST_LOG = TEST_ALIGN_DIR / "Log.final.out"
TEST_BAM = TEST_ALIGN_DIR / "Aligned.sortedByCoord.out.bam"

if TEST_LOG.is_file() and TEST_BAM.is_file():
    print("[OK] Test alignment already exists.")

elif TEST_ALIGN_DIR.exists() and any(TEST_ALIGN_DIR.iterdir()):
    raise FileExistsError(
        "Partial test alignment found. Inspect before rerunning."
    )

else:
    TEST_ALIGN_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(TEST_INDEX_DIR),
        "--readFilesIn", str(RAW_DIR / f"{TEST_SAMPLE}.fastq.gz"),
        "--readFilesCommand", "zcat",
        "--outSAMtype", "BAM", "SortedByCoordinate",
        "--outSAMstrandField", "intronMotif",
        "--outTmpDir", str(
            Path.home() / "star_tmp_final_project" /
            f"{TEST_SAMPLE}_sparse1_STARtmp"
        ),
        "--outFileNamePrefix", str(TEST_ALIGN_DIR) + "/"
    ]

    print("Running comparison alignment...", flush=True)
    subprocess.run(command, check=True)

# Read STAR alignment statistics
stats = {}

for line in TEST_LOG.read_text().splitlines():
    if "|" in line:
        key, value = line.split("|", 1)
        stats[key.strip()] = value.strip()

expected = {
    "Uniquely mapped reads number": 40076,
    "Number of reads mapped to multiple loci": 570,
    "Number of reads unmapped: too short": 1
}

print("\nSTAR alignment comparison — 1mo_Rep2")

for metric, expected_value in expected.items():
    observed = int(stats[metric].replace(",", ""))

    print(
        f"{metric}: "
        f"observed={observed}, "
        f"expected={expected_value}, "
        f"difference={observed - expected_value}"
    )

[OK] Test alignment already exists.

STAR alignment comparison — 1mo_Rep2
Uniquely mapped reads number: observed=40076, expected=40076, difference=0
Number of reads mapped to multiple loci: observed=570, expected=570, difference=0
Number of reads unmapped: too short: observed=1, expected=1, difference=0


In [19]:
# Align remaining samples using the validated STAR index
CORRECTED_STAR_DIR = STAR_DIR / "test_sparse1"

for sample in SAMPLES:
    sample_dir = CORRECTED_STAR_DIR / sample
    bam = sample_dir / "Aligned.sortedByCoord.out.bam"
    log = sample_dir / "Log.final.out"

    # Reuse complete and valid alignments
    if bam.is_file() and log.is_file():
        subprocess.run(
            ["samtools", "quickcheck", "-v", str(bam)],
            check=True
        )
        print(f"[OK] {sample}: existing alignment")
        continue

    if sample_dir.exists() and any(sample_dir.iterdir()):
        raise FileExistsError(
            f"Partial alignment detected: {sample_dir}"
        )

    sample_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(TEST_INDEX_DIR),
        "--readFilesIn", str(RAW_DIR / f"{sample}.fastq.gz"),
        "--readFilesCommand", "zcat",
        "--outSAMtype", "BAM", "SortedByCoordinate",
        "--outSAMstrandField", "intronMotif",
        "--outTmpDir", str(
            Path.home() / "star_tmp_final_project"
            / f"{sample}_sparse1_STARtmp"
        ),
        "--outFileNamePrefix", str(sample_dir) + "/"
    ]

    print(f"Aligning {sample}...", flush=True)
    subprocess.run(command, check=True)

    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True
    )

    if not log.is_file():
        raise FileNotFoundError(log)

    print(f"[OK] {sample}: alignment completed")

print("\nAll six corrected STAR alignments are available.")

[OK] 1mo_Rep1: existing alignment
[OK] 1mo_Rep2: existing alignment
[OK] 1mo_Rep3: existing alignment
[OK] 4mo_Rep1: existing alignment
[OK] 4mo_Rep2: existing alignment
[OK] 4mo_Rep3: existing alignment

All six corrected STAR alignments are available.


In [ ]:
# Verify reference gene counts and STAR alignment records
import re
import subprocess

# Count unique reference GENCODE genes
reference_genes = set()

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9:
            continue

        match = re.search(r'gene_id "([^"]+)"', fields[8])
        if match:
            reference_genes.add(match.group(1))

print("Q1 - Reference GENCODE genes:", len(reference_genes))

# Count BAM alignment records using corrected STAR output
corrected_bam = (
    CORRECTED_STAR_DIR
    / "1mo_Rep2"
    / "Aligned.sortedByCoord.out.bam"
)

result = subprocess.run(
    ["samtools", "view", "-c", str(corrected_bam)],
    capture_output=True,
    text=True,
    check=True
)

print("Q2 - BAM alignment records:", result.stdout.strip())

Q1 - Reference GENCODE genes: 167
Q2 - BAM alignment records: 41320


In [ ]:
# Count splice junction records from the corrected STAR alignment
JUNC_EXE = Path.home() / "miniconda3/envs/psiclass/bin/junc"

CORRECTED_BAM = (
    CORRECTED_STAR_DIR
    / "1mo_Rep2"
    / "Aligned.sortedByCoord.out.bam"
)

result = subprocess.run(
    [str(JUNC_EXE), str(CORRECTED_BAM), "-a"],
    capture_output=True,
    text=True,
    check=True
)

splice_records = len(result.stdout.splitlines())

print("BAM alignment records: 41320")
print("Splice junction records:", splice_records)

BAM alignment records: 41320
Splice junction records: 1099


In [ ]:
# Count alignment records in the corrected STAR BAM
bam = (
    CORRECTED_STAR_DIR
    / "1mo_Rep2"
    / "Aligned.sortedByCoord.out.bam"
)

subprocess.run(
    ["samtools", "view", "-c", str(bam)],
    check=True
)

41320


CompletedProcess(args=['samtools', 'view', '-c', '/mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/test_sparse1/1mo_Rep2/Aligned.sortedByCoord.out.bam'], returncode=0)

## 08. Transcript Assembly — PsiCLASS

Assemble transcripts from the six STAR-aligned RNA-seq samples using PsiCLASS.

The BAM files are provided through a sample list. PsiCLASS uses this list to reconstruct transcripts for individual samples and generate a consensus assembly.

PsiCLASS is executed from its dedicated Conda environment.

In [23]:
# Prepare BAM files for PsiCLASS
BAM_FILES = []

for sample in SAMPLES:
    bam, log = star_files(sample)

    if not bam.is_file():
        raise FileNotFoundError(f"Missing BAM: {bam}")

    BAM_FILES.append(bam.resolve())

# Create PsiCLASS directory
PSICLASS_DIR.mkdir(parents=True, exist_ok=True)

# Prepare BAM list
bamlist_content = "\n".join(str(bam) for bam in BAM_FILES) + "\n"

if BAMLIST.exists():
    if BAMLIST.read_text() != bamlist_content:
        raise ValueError("Existing BAM list differs from the current samples.")
    print("[OK] Existing BAM list verified.")
else:
    BAMLIST.write_text(bamlist_content)
    print("[OK] BAM list created.")

# Check PsiCLASS executable
if not PSICLASS_EXE.is_file():
    raise FileNotFoundError(f"PsiCLASS executable not found: {PSICLASS_EXE}")

print("[OK] PsiCLASS executable found.")

# Display input sample order
print("\nPsiCLASS input files:")

for i, bam in enumerate(BAM_FILES):
    print(f"{i}: {bam.parent.name} / {bam.name}")

# Check existing assembly
print("\nConsensus assembly:")
print("FOUND" if PSICLASS_VOTE.is_file() else "NOT GENERATED")

[OK] Existing BAM list verified.
[OK] PsiCLASS executable found.

PsiCLASS input files:
0: 1mo_Rep1 / Aligned.sortedByCoord.out.bam
1: 1mo_Rep2 / Aligned.sortedByCoord.out.bam
2: 1mo_Rep3 / Aligned.sortedByCoord.out.bam
3: 4mo_Rep1 / Aligned.sortedByCoord.out.bam
4: 4mo_Rep2 / Aligned.sortedByCoord.out.bam
5: 4mo_Rep3 / Aligned.sortedByCoord.out.bam

Consensus assembly:
FOUND


In [24]:
# Run PsiCLASS transcript assembly
OUTPUT_PREFIX = PSICLASS_DIR / "psiclass"
VOTE_GTF = PSICLASS_DIR / "psiclass_vote.gtf"
PSICLASS_LOG = PSICLASS_DIR / "psiclass.log"

# Protect existing results
if VOTE_GTF.exists():
    print("[OK] PsiCLASS consensus assembly already exists.")

else:
    existing_outputs = list(PSICLASS_DIR.glob("psiclass_sample_*.gtf"))

    if existing_outputs or PSICLASS_LOG.exists():
        raise FileExistsError(
            "Existing PsiCLASS files detected. "
            "Inspect them before rerunning."
        )

    # Define PsiCLASS command
    command = [
        str(PSICLASS_EXE),
        "--lb", str(BAMLIST),
        "-o", str(OUTPUT_PREFIX),
        "-p", str(THREADS)
    ]

    # Execute and save log
    print("Running PsiCLASS transcript assembly...", flush=True)

    with PSICLASS_LOG.open("w") as log:
        subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True
        )

# Verify consensus assembly
if not VOTE_GTF.is_file() or VOTE_GTF.stat().st_size == 0:
    raise FileNotFoundError(
        f"PsiCLASS consensus assembly not found: {VOTE_GTF}"
    )

print("\n[OK] PsiCLASS consensus assembly:", VOTE_GTF)

# Check individual sample assemblies
for i in range(len(SAMPLES)):
    sample_gtf = PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    status = "FOUND" if sample_gtf.is_file() else "MISSING"
    print(f"psiclass_sample_{i}.gtf: {status}")

[OK] PsiCLASS consensus assembly already exists.

[OK] PsiCLASS consensus assembly: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS/psiclass_vote.gtf
psiclass_sample_0.gtf: FOUND
psiclass_sample_1.gtf: FOUND
psiclass_sample_2.gtf: FOUND
psiclass_sample_3.gtf: FOUND
psiclass_sample_4.gtf: FOUND
psiclass_sample_5.gtf: FOUND


In [25]:
# Assemble transcripts from corrected STAR alignments
CORRECTED_STAR_DIR = STAR_DIR / "test_sparse1"
CORRECTED_PSICLASS_DIR = RESULTS_DIR / "PsiCLASS_corrected"

CORRECTED_BAMLIST = CORRECTED_PSICLASS_DIR / "bamlist.psiclass"
CORRECTED_PSICLASS_PREFIX = CORRECTED_PSICLASS_DIR / "psiclass"
CORRECTED_PSICLASS_VOTE = (
    CORRECTED_PSICLASS_DIR / "psiclass_vote.gtf"
)

# Collect corrected BAM files in sample order
corrected_bams = [
    CORRECTED_STAR_DIR / sample / "Aligned.sortedByCoord.out.bam"
    for sample in SAMPLES
]

for sample, bam in zip(SAMPLES, corrected_bams):
    if not bam.is_file():
        raise FileNotFoundError(bam)

    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True
    )

    print(f"[OK] {sample}")

CORRECTED_PSICLASS_DIR.mkdir(parents=True, exist_ok=True)

# Prepare PsiCLASS BAM list
bamlist_content = "\n".join(map(str, corrected_bams)) + "\n"

if CORRECTED_BAMLIST.exists():
    if CORRECTED_BAMLIST.read_text() != bamlist_content:
        raise ValueError("Existing corrected BAM list differs.")
else:
    CORRECTED_BAMLIST.write_text(bamlist_content)

# Run PsiCLASS without overwriting previous results
if CORRECTED_PSICLASS_VOTE.is_file() and (
    CORRECTED_PSICLASS_VOTE.stat().st_size > 0
):
    print("\n[OK] Corrected PsiCLASS assembly already exists.")

elif any(CORRECTED_PSICLASS_DIR.glob("psiclass_*.gtf")):
    raise FileExistsError(
        "Partial PsiCLASS outputs detected. Inspect before rerunning."
    )

else:
    command = [
        str(PSICLASS_EXE),
        "--lb", str(CORRECTED_BAMLIST),
        "-o", str(CORRECTED_PSICLASS_PREFIX),
        "-p", str(THREADS)
    ]

    print("\nRunning PsiCLASS with corrected BAMs...", flush=True)
    subprocess.run(command, check=True)

# Verify outputs
expected_gtfs = [CORRECTED_PSICLASS_VOTE] + [
    CORRECTED_PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

for gtf in expected_gtfs:
    if not gtf.is_file() or gtf.stat().st_size == 0:
        raise FileNotFoundError(gtf)

    print(f"[OK] {gtf.name}")

print("\nCorrected PsiCLASS assembly completed.")

[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

[OK] Corrected PsiCLASS assembly already exists.
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf

Corrected PsiCLASS assembly completed.


In [26]:
# Summarize corrected PsiCLASS transcript assemblies
gtf_files = [
    ("Consensus", CORRECTED_PSICLASS_VOTE),
    *[
        (
            sample,
            CORRECTED_PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
        )
        for i, sample in enumerate(SAMPLES)
    ]
]

assembly_summary = []

for sample, gtf_file in gtf_files:
    gene_ids = set()
    transcript_ids = set()
    exon_counts = {}

    with gtf_file.open() as gtf:
        for line in gtf:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            attrs = dict(
                re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
            )

            gene_id = attrs.get("gene_id")
            transcript_id = attrs.get("transcript_id")

            if gene_id:
                gene_ids.add(gene_id)

            if transcript_id:
                transcript_ids.add(transcript_id)

                if fields[2] == "exon":
                    exon_counts[transcript_id] = (
                        exon_counts.get(transcript_id, 0) + 1
                    )

    assembly_summary.append({
        "Assembly": sample,
        "Genes (loci)": len(gene_ids),
        "Transcripts": len(transcript_ids),
        "Single-exon transcripts": sum(
            count == 1 for count in exon_counts.values()
        )
    })

assembly_summary = pd.DataFrame(assembly_summary)

display(assembly_summary)

,Assembly,Genes (loci),Transcripts,Single-exon transcripts
0,Consensus,168,211,84
1,1mo_Rep1,173,204,85
2,1mo_Rep2,171,210,84
3,1mo_Rep3,173,212,86
4,4mo_Rep1,173,209,85
5,4mo_Rep2,172,211,84
6,4mo_Rep3,173,212,86


In [27]:
# Summarize PsiCLASS transcript assemblies
def summarize_gtf(gtf_file):
    genes = defaultdict(set)
    exon_counts = defaultdict(int)

    with gtf_file.open() as file:
        for line in file:
            if line.startswith("#") or not line.strip():
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            attributes = fields[8]
            gene = re.search(r'gene_id "([^"]+)"', attributes)
            transcript = re.search(r'transcript_id "([^"]+)"', attributes)

            if gene and transcript:
                gene_id = gene.group(1)
                transcript_id = transcript.group(1)

                genes[gene_id].add(transcript_id)

                if fields[2] == "exon":
                    exon_counts[transcript_id] += 1

    return {
        "Genes": len(genes),
        "Transcripts": sum(len(t) for t in genes.values()),
        "Single-exon transcripts": sum(
            count == 1 for count in exon_counts.values()
        ),
        "Multi-exon transcripts": sum(
            count > 1 for count in exon_counts.values()
        )
    }


# Collect assembly statistics
gtf_files = [PSICLASS_VOTE] + [
    PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

assembly_summary = []

for gtf in gtf_files:
    if not gtf.is_file():
        raise FileNotFoundError(gtf)

    stats = summarize_gtf(gtf)
    assembly_summary.append({"Assembly": gtf.name, **stats})

assembly_df = pd.DataFrame(assembly_summary)
display(assembly_df)

,Assembly,Genes,Transcripts,Single-exon transcripts,Multi-exon transcripts
0,psiclass_vote.gtf,171,223,87,136
1,psiclass_sample_0.gtf,172,222,88,134
2,psiclass_sample_1.gtf,169,222,87,135
3,psiclass_sample_2.gtf,170,220,88,132
4,psiclass_sample_3.gtf,172,217,88,129
5,psiclass_sample_4.gtf,170,221,88,133
6,psiclass_sample_5.gtf,172,221,88,133


In [53]:
# Verify sample-specific transcript counts using the course method

sample_gtf = (
    CORRECTED_PSICLASS_DIR / "psiclass_sample_1.gtf"
)

# Reproduce the course command
command = (
    f"cut -f9 '{sample_gtf}' "
    "| cut -d ' ' -f4 "
    "| sort -u "
    "| wc -l"
)

result = subprocess.run(
    ["bash", "-o", "pipefail", "-c", command],
    capture_output=True,
    text=True,
    check=True
)

print("Course method:", result.stdout.strip())

# Independent count of transcript IDs
transcript_ids = set()

with sample_gtf.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        match = re.search(
            r'transcript_id "([^"]+)"',
            fields[8]
        )

        if match:
            transcript_ids.add(match.group(1))

print("Unique transcript IDs:", len(transcript_ids))

Course method: 213
Unique transcript IDs: 210


In [54]:
# Compare transcript identifiers extracted by both methods

result = subprocess.run(
    [
        "bash", "-o", "pipefail", "-c",
        f"cut -f9 '{sample_gtf}' | cut -d ' ' -f4 | sort -u"
    ],
    capture_output=True,
    text=True,
    check=True
)

course_values = set(result.stdout.splitlines())

print("Values counted by the course method but not by Python:")
for value in sorted(course_values - transcript_ids):
    print(repr(value))

print("\nPython transcript IDs absent from the course method:")
for value in sorted(transcript_ids - course_values):
    print(repr(value))

Values counted by the course method but not by Python:
'"7.100.0";'
'"7.100.2";'
'"7.103.0";'
'"7.104.0";'
'"7.105.0";'
'"7.107.0";'
'"7.109.0";'
'"7.115.0";'
'"7.118.0";'
'"7.124.0";'
'"7.127.0";'
'"7.127.1";'
'"7.139.0";'
'"7.143.1";'
'"7.143.2";'
'"7.144.0";'
'"7.145.0";'
'"7.15.0";'
'"7.153.0";'
'"7.155.0";'
'"7.156.0";'
'"7.157.0";'
'"7.158.0";'
'"7.159.0";'
'"7.161.0";'
'"7.166.0";'
'"7.168.0";'
'"7.169.0";'
'"7.17.0";'
'"7.170.0";'
'"7.171.0";'
'"7.174.0";'
'"7.177.0";'
'"7.18.0";'
'"7.184.0";'
'"7.184.1";'
'"7.193.0";'
'"7.200.0";'
'"7.21.0";'
'"7.215.0";'
'"7.216.0";'
'"7.217.0";'
'"7.218.0";'
'"7.219.0";'
'"7.22.0";'
'"7.221.0";'
'"7.222.0";'
'"7.223.0";'
'"7.227.0";'
'"7.228.0";'
'"7.229.0";'
'"7.23.0";'
'"7.23.1";'
'"7.23.2";'
'"7.23.4";'
'"7.23.5";'
'"7.23.7";'
'"7.230.0";'
'"7.232.0";'
'"7.233.0";'
'"7.234.0";'
'"7.235.0";'
'"7.237.0";'
'"7.239.0";'
'"7.240.0";'
'"7.241.0";'
'"7.243.0";'
'"7.245.0";'
'"7.246.0";'
'"7.248.0";'
'"7.249.0";'
'"7.254.0";'
'"7.254.1";'
'"7.258

## 09. Gene Annotation

Assign GENCODE vM30 gene names and genomic strand information to the PsiCLASS transcript assemblies.

The annotation follows the course workflow using the original `add-genename` and `add-strand` utilities.

**Methods:**
1. Prepare a list containing the consensus and six sample-specific assemblies.
2. Assign reference gene names using `add-genename`.
3. Assign genomic strand information using `add-strand`.
4. Validate the annotated consensus assembly.

Original PsiCLASS GTF files are preserved. Annotated files are saved in `WithGeneNames`.

In [28]:
# Prepare annotation inputs

ADD_GENENAME = (
    Path.home() / "miniconda3/envs/psiclass/bin/add-genename"
)
ADD_STRAND = REPO_DIR.parent / "docs" / "add-strand"

ANNOTATION_GTF_LIST = PSICLASS_DIR / "psiclass_annotation.list"

# Consensus and individual assemblies
annotation_gtfs = [PSICLASS_VOTE] + [
    PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

# Verify input files and utilities
for path in [
    GENCODE_GTF,
    ADD_GENENAME,
    ADD_STRAND,
    *annotation_gtfs
]:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")

# Save the annotation input list
list_content = "\n".join(
    str(gtf.resolve()) for gtf in annotation_gtfs
) + "\n"

if ANNOTATION_GTF_LIST.exists():
    if ANNOTATION_GTF_LIST.read_text() != list_content:
        raise ValueError("Existing annotation list differs.")
else:
    ANNOTATION_GTF_LIST.write_text(list_content)

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

print("\n[OK] Annotation list prepared.")
print("Assemblies:", len(annotation_gtfs))

[OK] gencode.vM30.region.gtf
[OK] add-genename
[OK] add-strand
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf

[OK] Annotation list prepared.
Assemblies: 7


In [29]:
# Inspect existing gene annotation outputs
print("Annotation directory:", ANNOTATION_DIR)
print("\nExpected GTF files:")

for path in expected_gtfs if "expected_gtfs" in globals() else [
    ANNOTATION_DIR / gtf.name for gtf in annotation_gtfs
]:
    if path.is_file():
        size = path.stat().st_size
        print(f"[FOUND] {path.name} — {size:,} bytes")
    else:
        print(f"[MISSING] {path.name}")

print("\nOther files in the annotation directory:")

if ANNOTATION_DIR.is_dir():
    for path in sorted(ANNOTATION_DIR.iterdir()):
        print(path.name)

Annotation directory: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS/WithGeneNames

Expected GTF files:
[FOUND] psiclass_vote.gtf — 202,033 bytes
[FOUND] psiclass_sample_0.gtf — 174,125 bytes
[FOUND] psiclass_sample_1.gtf — 177,599 bytes
[FOUND] psiclass_sample_2.gtf — 183,392 bytes
[FOUND] psiclass_sample_3.gtf — 177,552 bytes
[FOUND] psiclass_sample_4.gtf — 181,275 bytes
[FOUND] psiclass_sample_5.gtf — 183,073 bytes

Other files in the annotation directory:
psiclass_sample_0.gtf
psiclass_sample_1.gtf
psiclass_sample_2.gtf
psiclass_sample_3.gtf
psiclass_sample_4.gtf
psiclass_sample_5.gtf
psiclass_vote.gtf
psiclass_vote.withStrand.gtf


In [30]:
# Assign GENCODE gene names

ANNOTATED_VOTE = ANNOTATION_DIR / "psiclass_vote.gtf"

# Check existing sample annotations
sample_gtfs = [
    ANNOTATION_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

for gtf in sample_gtfs:
    if not gtf.is_file() or gtf.stat().st_size == 0:
        raise FileNotFoundError(f"Missing sample annotation: {gtf}")

print("[OK] Six sample annotations found.")

# Annotate the consensus assembly separately
if ANNOTATED_VOTE.is_file() and ANNOTATED_VOTE.stat().st_size > 0:
    print("[OK] Consensus annotation already exists.")

elif ANNOTATED_VOTE.exists():
    raise FileExistsError(
        f"Existing consensus annotation needs inspection: {ANNOTATED_VOTE}"
    )

else:
    # Create a list containing only the consensus assembly
    vote_list = PSICLASS_DIR / "psiclass_vote_annotation.list"
    vote_content = str(PSICLASS_VOTE.resolve()) + "\n"

    if vote_list.exists():
        if vote_list.read_text() != vote_content:
            raise ValueError("Existing consensus annotation list differs.")
    else:
        vote_list.write_text(vote_content)

    command = [
        str(ADD_GENENAME),
        str(GENCODE_GTF),
        str(vote_list),
        "-o", str(ANNOTATION_DIR)
    ]

    print("Annotating PsiCLASS consensus assembly...", flush=True)

    subprocess.run(command, check=True)

# Verify annotated consensus
if not ANNOTATED_VOTE.is_file() or ANNOTATED_VOTE.stat().st_size == 0:
    raise FileNotFoundError(
        f"Consensus annotation was not generated: {ANNOTATED_VOTE}"
    )

print("\n[OK] Annotated consensus:", ANNOTATED_VOTE)

[OK] Six sample annotations found.
[OK] Consensus annotation already exists.

[OK] Annotated consensus: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS/WithGeneNames/psiclass_vote.gtf


In [31]:
# Assign genomic strand information
FINAL_GTF = ANNOTATION_DIR / "psiclass_vote.withStrand.gtf"

if FINAL_GTF.is_file() and FINAL_GTF.stat().st_size > 0:
    print("[OK] Strand-annotated GTF already exists.")

elif FINAL_GTF.exists():
    raise FileExistsError(
        f"Existing output needs inspection: {FINAL_GTF}"
    )

else:
    # Follow the course annotation procedure
    command = [
        "perl",
        str(ADD_STRAND),
        str(GENCODE_GTF),
        "-r"
    ]

    print("Assigning genomic strand information...", flush=True)

    # Read annotated GTF from stdin and write output
    with ANNOTATED_VOTE.open("r") as source:
        with FINAL_GTF.open("x") as output:
            result = subprocess.run(
                command,
                stdin=source,
                stdout=output,
                stderr=subprocess.PIPE,
                text=True
            )

    if result.returncode != 0:
        raise RuntimeError(
            f"add-strand failed:\n{result.stderr}\n"
            f"Inspect the output: {FINAL_GTF}"
        )

# Validate final annotation
genes = set()
transcripts = set()
strand_counts = {"+": 0, "-": 0, ".": 0}
records = 0
named_records = 0

with FINAL_GTF.open() as file:
    for line in file:
        if line.startswith("#") or not line.strip():
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            raise ValueError("Invalid GTF record.")

        records += 1
        strand = fields[6]

        if strand not in strand_counts:
            raise ValueError(f"Unexpected strand: {strand}")

        strand_counts[strand] += 1

        attributes = fields[8]

        gene = re.search(r'gene_id\s+"([^"]+)"', attributes)
        transcript = re.search(r'transcript_id\s+"([^"]+)"', attributes)
        name = re.search(r'gene_name\s+"([^"]+)"', attributes)

        if gene:
            genes.add(gene.group(1))

        if transcript:
            transcripts.add(transcript.group(1))

        if name:
            named_records += 1

if records == 0:
    raise ValueError("The final annotation is empty.")

print("\nFinal PsiCLASS annotation:")
print("GTF records:", records)
print("Genes:", len(genes))
print("Transcripts:", len(transcripts))
print("Records with gene_name:", named_records)
print("Positive strand:", strand_counts["+"])
print("Negative strand:", strand_counts["-"])
print("Unknown strand:", strand_counts["."])
print("\nOutput:", FINAL_GTF)

[OK] Strand-annotated GTF already exists.

Final PsiCLASS annotation:
GTF records: 1136
Genes: 145
Transcripts: 197
Records with gene_name: 1136
Positive strand: 400
Negative strand: 736
Unknown strand: 0

Output: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS/WithGeneNames/psiclass_vote.withStrand.gtf


In [32]:
# Assess reference gene identification and transcript exon structure
# Collect reference GENCODE gene names
reference_names = set()

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "gene":
            continue

        match = re.search(r'gene_name "([^"]+)"', fields[8])

        if match:
            reference_names.add(match.group(1))

# Inspect final PsiCLASS annotation
identified_names = set()
exon_counts = Counter()

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        attributes = fields[8]

        gene_name = re.search(r'gene_name "([^"]+)"', attributes)
        transcript = re.search(r'transcript_id "([^"]+)"', attributes)

        if gene_name:
            identified_names.add(gene_name.group(1))

        if fields[2] == "exon" and transcript:
            exon_counts[transcript.group(1)] += 1

matched_names = identified_names & reference_names
single_exon = sum(n == 1 for n in exon_counts.values())

print("Reference GENCODE genes identified:", len(matched_names))
print("Single-exon transcripts in final annotation:", single_exon)

Reference GENCODE genes identified: 73
Single-exon transcripts in final annotation: 61


In [33]:
# Assign GENCODE gene names to corrected PsiCLASS assemblies
CORRECTED_ANNOTATION_DIR = (
    CORRECTED_PSICLASS_DIR / "WithGeneNames"
)

CORRECTED_ANNOTATION_LIST = (
    CORRECTED_PSICLASS_DIR / "psiclass_annotation.list"
)

corrected_gtfs = [CORRECTED_PSICLASS_VOTE] + [
    CORRECTED_PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

# Verify annotation inputs
for path in [GENCODE_GTF, ADD_GENENAME, *corrected_gtfs]:
    if not path.is_file():
        raise FileNotFoundError(path)

# Prepare the list of assemblies
list_content = "\n".join(
    str(path.resolve()) for path in corrected_gtfs
) + "\n"

if CORRECTED_ANNOTATION_LIST.exists():
    if CORRECTED_ANNOTATION_LIST.read_text() != list_content:
        raise ValueError("Existing annotation list differs.")
else:
    CORRECTED_ANNOTATION_LIST.write_text(list_content)

CORRECTED_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

expected_annotations = [
    CORRECTED_ANNOTATION_DIR / path.name
    for path in corrected_gtfs
]

if all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_annotations
):
    print("[OK] Corrected gene annotations already exist.")

elif any(path.exists() for path in expected_annotations):
    raise FileExistsError(
        "Partial annotation outputs detected. Inspect before rerunning."
    )

else:
    command = [
        str(ADD_GENENAME),
        str(GENCODE_GTF),
        str(CORRECTED_ANNOTATION_LIST),
        "-o", str(CORRECTED_ANNOTATION_DIR)
    ]

    print("Assigning GENCODE gene names...", flush=True)
    subprocess.run(command, check=True)

print("\nCorrected gene-name annotations:")

for path in expected_annotations:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)

    print(f"[OK] {path.name}")

[OK] Corrected gene annotations already exist.

Corrected gene-name annotations:
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf


In [34]:
# Assign genomic strand information to the corrected PsiCLASS assembly
CORRECTED_ANNOTATED_VOTE = (
    CORRECTED_ANNOTATION_DIR / "psiclass_vote.gtf"
)

CORRECTED_FINAL_GTF = (
    CORRECTED_ANNOTATION_DIR / "psiclass_vote.withStrand.gtf"
)

# Verify inputs
for path in [GENCODE_GTF, ADD_STRAND, CORRECTED_ANNOTATED_VOTE]:
    if not path.is_file():
        raise FileNotFoundError(path)

# Preserve existing output
if CORRECTED_FINAL_GTF.is_file() and CORRECTED_FINAL_GTF.stat().st_size > 0:
    print("[OK] Corrected strand annotation already exists.")

elif CORRECTED_FINAL_GTF.exists():
    raise FileExistsError(
        f"Empty or partial annotation: {CORRECTED_FINAL_GTF}"
    )

else:
    command = [
        "perl",
        str(ADD_STRAND),
        str(GENCODE_GTF),
        "-r"
    ]

    print("Assigning genomic strand information...", flush=True)

    # Write to a temporary file and rename only after success
    temp_gtf = CORRECTED_FINAL_GTF.with_suffix(".gtf.tmp")

    if temp_gtf.exists():
        raise FileExistsError(f"Temporary annotation already exists: {temp_gtf}")

    try:
        with CORRECTED_ANNOTATED_VOTE.open("r") as source:
            with temp_gtf.open("x") as output:
                subprocess.run(
                    command,
                    stdin=source,
                    stdout=output,
                    check=True
                )

        if temp_gtf.stat().st_size == 0:
            raise ValueError("Strand annotation output is empty.")

        temp_gtf.rename(CORRECTED_FINAL_GTF)

    except Exception:
        print(f"Inspect temporary output if present: {temp_gtf}")
        raise

# Validate final GTF
gene_ids = set()
transcript_ids = set()
strands = set()
records = 0

with CORRECTED_FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            raise ValueError("Invalid GTF record.")

        records += 1
        strands.add(fields[6])

        attrs = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        if attrs.get("gene_id"):
            gene_ids.add(attrs["gene_id"])

        if attrs.get("transcript_id"):
            transcript_ids.add(attrs["transcript_id"])

print("\nCorrected final PsiCLASS annotation:")
print("GTF records:", records)
print("Genes (loci):", len(gene_ids))
print("Transcripts:", len(transcript_ids))
print("Strands:", ", ".join(sorted(strands)))
print("Output:", CORRECTED_FINAL_GTF)

[OK] Corrected strand annotation already exists.

Corrected final PsiCLASS annotation:
GTF records: 1155
Genes (loci): 142
Transcripts: 185
Strands: +, -
Output: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS_corrected/WithGeneNames/psiclass_vote.withStrand.gtf


In [35]:
# Assess reference gene identification and transcript exon structure
reference_names = set()

# Collect GENCODE reference gene names
with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "gene":
            continue

        match = re.search(r'gene_name "([^"]+)"', fields[8])

        if match:
            reference_names.add(match.group(1))

# Inspect corrected PsiCLASS annotation
identified_names = set()
exon_counts = Counter()
transcript_ids = set()

with CORRECTED_FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        attrs = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        gene_name = attrs.get("gene_name")
        transcript_id = attrs.get("transcript_id")

        if fields[2] == "transcript":
            if gene_name and gene_name != "novel":
                identified_names.add(gene_name)

            if transcript_id:
                transcript_ids.add(transcript_id)

        elif fields[2] == "exon" and transcript_id:
            exon_counts[transcript_id] += 1

# Count reference genes and single-exon transcripts
matched_names = identified_names & reference_names
single_exon = sum(
    exon_counts[transcript_id] == 1
    for transcript_id in transcript_ids
)

print("Reference GENCODE genes identified:", len(matched_names))
print("Single-exon transcripts:", single_exon)
print("Total transcripts:", len(transcript_ids))

Reference GENCODE genes identified: 75
Single-exon transcripts: 58
Total transcripts: 185


## 10. Differential Gene Expression — Cuffdiff2

Compare gene expression between 1-month-old and 4-month-old mouse hippocampus samples using Cuffdiff2.

The analysis uses three biological replicates per condition and the annotated PsiCLASS consensus assembly.

Cuffdiff2 estimates expression in FPKM and performs statistical tests for differential expression. Results include unadjusted p-values and FDR-adjusted q-values.

In [36]:
# Verify Cuffdiff2 input files
FINAL_GTF = ANNOTATION_DIR / "psiclass_vote.withStrand.gtf"

control_bams = [star_files(s)[0] for s in SAMPLES_1MO]
test_bams = [star_files(s)[0] for s in SAMPLES_4MO]

# Check annotation
if not FINAL_GTF.is_file():
    raise FileNotFoundError(FINAL_GTF)

print("[OK] Final PsiCLASS annotation")

# Check BAM files
for bam in control_bams + test_bams:
    if not bam.is_file():
        raise FileNotFoundError(bam)

    print(f"[OK] {bam.parent.name}")

print("\nAll Cuffdiff2 input files are available.")

[OK] Final PsiCLASS annotation
[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

All Cuffdiff2 input files are available.


In [37]:
# Run Cuffdiff2 differential expression analysis
CUFFDIFF_LOG = CUFFDIFF_DIR / "cuffdiff.log"
GENE_DIFF = CUFFDIFF_DIR / "gene_exp.diff"

# Prepare biological replicate groups
control_group = ",".join(str(bam) for bam in control_bams)
test_group = ",".join(str(bam) for bam in test_bams)

# Check existing results
if GENE_DIFF.is_file() and GENE_DIFF.stat().st_size > 0:
    print("[OK] Existing Cuffdiff2 results found.")

elif CUFFDIFF_DIR.exists() and any(CUFFDIFF_DIR.iterdir()):
    raise FileExistsError(
        "Cuffdiff2 directory contains existing files. "
        "Inspect them before rerunning."
    )

else:
    CUFFDIFF_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "cuffdiff",
        "-o", str(CUFFDIFF_DIR),
        "-p", str(THREADS),
        "-L", "1mo,4mo",
        str(FINAL_GTF),
        control_group,
        test_group
    ]

    print("Running Cuffdiff2...", flush=True)

    with CUFFDIFF_LOG.open("w") as log:
        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT
        )

    if result.returncode != 0:
        print(CUFFDIFF_LOG.read_text(errors="replace")[-3000:])
        raise RuntimeError(
            "Cuffdiff2 failed. Check the execution log."
        )

# Verify main output files
expected_outputs = [
    CUFFDIFF_DIR / "gene_exp.diff",
    CUFFDIFF_DIR / "genes.fpkm_tracking"
]

print("\nCuffdiff2 output verification:")

for file in expected_outputs:
    if not file.is_file() or file.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty output: {file}")

    print(f"[OK] {file.name}")

print("\nCuffdiff2 analysis completed.")

[OK] Existing Cuffdiff2 results found.

Cuffdiff2 output verification:
[OK] gene_exp.diff
[OK] genes.fpkm_tracking

Cuffdiff2 analysis completed.


In [38]:
# Summarize Cuffdiff2 differential expression results
# Load gene-level results
cuff_results = pd.read_csv(GENE_DIFF, sep="\t")

print("Cuffdiff2 result columns:")
print(cuff_results.columns.tolist())

# Convert statistical and expression columns
for column in ["p_value", "q_value", "value_1", "value_2"]:
    cuff_results[column] = pd.to_numeric(
        cuff_results[column], errors="coerce"
    )

# Identify significant genes
significant = cuff_results["significant"].astype(str).str.lower().eq("yes")

# Alternative significance criterion
logfc = pd.to_numeric(
    cuff_results["log2(fold_change)"], errors="coerce"
)

pval_fc = (
    cuff_results["status"].eq("OK")
    & (cuff_results["p_value"] <= 0.05)
    & (logfc.abs() >= 1)
)

# Expression threshold
expressed = (
    (cuff_results["value_1"] >= 2)
    | (cuff_results["value_2"] >= 2)
)

# Display summary
summary = pd.DataFrame({
    "Metric": [
        "Total gene-level entries",
        "Significant entries (Cuffdiff2)",
        "p-value <= 0.05 and |log2FC| >= 1",
        "FPKM >= 2 in at least one condition"
    ],
    "Count": [
        len(cuff_results),
        int(significant.sum()),
        int(pval_fc.sum()),
        int(expressed.sum())
    ]
})

display(summary)

Cuffdiff2 result columns:
['test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2', 'status', 'value_1', 'value_2', 'log2(fold_change)', 'test_stat', 'p_value', 'q_value', 'significant']


,Metric,Count
0,Total gene-level entries,145
1,Significant entries (Cuffdiff2),4
2,p-value <= 0.05 and |log2FC| >= 1,2
3,FPKM >= 2 in at least one condition,145


In [39]:
# Identify reference GENCODE genes among significant Cuffdiff2 features
# Collect reference gene names
reference_names = set()

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9 or fields[2] != "gene":
            continue

        match = re.search(r'gene_name "([^"]+)"', fields[8])
        if match:
            reference_names.add(match.group(1))

# Map PsiCLASS IDs to reference gene names
gene_mapping = defaultdict(set)

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attrs = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))

        gene_id = attrs.get("gene_id")
        gene_name = attrs.get("gene_name")

        if gene_id and gene_name in reference_names:
            gene_mapping[gene_id].add(gene_name)

# Identify significant PsiCLASS features
cuff_sig = cuff_results.loc[
    cuff_results["significant"].astype(str).str.lower().eq("yes"),
    "gene_id"
].astype(str).unique()

identified = set()

for gene_id in cuff_sig:
    names = gene_mapping.get(gene_id, set())
    identified.update(names)
    print(f"{gene_id}: {', '.join(sorted(names)) or 'No reference match'}")

print("\nDistinct reference GENCODE genes:", len(identified))

7.17: Arhgap33
7.501: Sirt2
7.8: Mrps12
7.86: Sirt2

Distinct reference GENCODE genes: 3


In [40]:
# Run Cuffdiff2 using corrected STAR alignments and PsiCLASS annotations

CORRECTED_CUFFDIFF_DIR = RESULTS_DIR / "Cuffdiff_corrected"
CORRECTED_GENE_DIFF = CORRECTED_CUFFDIFF_DIR / "gene_exp.diff"
CORRECTED_CUFFDIFF_LOG = CORRECTED_CUFFDIFF_DIR / "cuffdiff.log"

# Prepare biological replicate groups
corrected_control = ",".join(
    str(CORRECTED_STAR_DIR / sample / "Aligned.sortedByCoord.out.bam")
    for sample in SAMPLES_1MO
)

corrected_test = ",".join(
    str(CORRECTED_STAR_DIR / sample / "Aligned.sortedByCoord.out.bam")
    for sample in SAMPLES_4MO
)

# Verify input annotation
if not CORRECTED_FINAL_GTF.is_file():
    raise FileNotFoundError(CORRECTED_FINAL_GTF)

# Preserve existing results
if CORRECTED_GENE_DIFF.is_file() and CORRECTED_GENE_DIFF.stat().st_size > 0:
    print("[OK] Corrected Cuffdiff2 results already exist.")

elif CORRECTED_CUFFDIFF_DIR.exists() and any(
    CORRECTED_CUFFDIFF_DIR.iterdir()
):
    raise FileExistsError(
        "Partial Cuffdiff2 outputs detected. Inspect before rerunning."
    )

else:
    CORRECTED_CUFFDIFF_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "cuffdiff",
        "-o", str(CORRECTED_CUFFDIFF_DIR),
        "-p", str(THREADS),
        "-L", "1mo,4mo",
        str(CORRECTED_FINAL_GTF),
        corrected_control,
        corrected_test
    ]

    print("Running corrected Cuffdiff2 analysis...", flush=True)

    with CORRECTED_CUFFDIFF_LOG.open("w") as log:
        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT
        )

    if result.returncode != 0:
        print(CORRECTED_CUFFDIFF_LOG.read_text(errors="replace")[-3000:])
        raise RuntimeError("Corrected Cuffdiff2 analysis failed.")

# Verify main results
for filename in ["gene_exp.diff", "genes.fpkm_tracking"]:
    path = CORRECTED_CUFFDIFF_DIR / filename

    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)

    print(f"[OK] {filename}")

print("\nCorrected Cuffdiff2 analysis completed.")

[OK] Corrected Cuffdiff2 results already exist.
[OK] gene_exp.diff
[OK] genes.fpkm_tracking

Corrected Cuffdiff2 analysis completed.


In [41]:
# Assess differential expression and FPKM thresholds

corrected_cuff = pd.read_csv(
    CORRECTED_GENE_DIFF,
    sep="\t",
    dtype={"gene_id": str}
)

for col in [
    "p_value", "q_value",
    "log2(fold_change)", "value_1", "value_2"
]:
    corrected_cuff[col] = pd.to_numeric(
        corrected_cuff[col], errors="coerce"
    )

significant = (
    corrected_cuff["significant"]
    .astype(str).str.lower().eq("yes")
)

pval_fc = (
    corrected_cuff["p_value"].le(0.05)
    & corrected_cuff["log2(fold_change)"].abs().ge(1)
)

expressed = (
    corrected_cuff["value_1"].ge(2)
    | corrected_cuff["value_2"].ge(2)
)

print("Significant Cuffdiff2 entries:", int(significant.sum()))
print("p-value <= 0.05 and |log2FC| >= 1:", int(pval_fc.sum()))
print("FPKM >= 2 in at least one condition:", int(expressed.sum()))

Significant Cuffdiff2 entries: 4
p-value <= 0.05 and |log2FC| >= 1: 1
FPKM >= 2 in at least one condition: 142


In [42]:
# Identify reference GENCODE genes among significant Cuffdiff2 features
gene_mapping = defaultdict(set)

with CORRECTED_FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attrs = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        gene_id = attrs.get("gene_id")
        gene_name = attrs.get("gene_name")

        if gene_id and gene_name and "novel" not in gene_name.lower():
            gene_mapping[gene_id].add(gene_name)

significant_ids = corrected_cuff.loc[
    corrected_cuff["significant"].astype(str).str.lower().eq("yes"),
    "gene_id"
].astype(str).unique()

reference_genes = set()

for gene_id in significant_ids:
    names = gene_mapping.get(gene_id, set())
    reference_genes.update(names)

    print(
        f"{gene_id}: {', '.join(sorted(names)) or 'No reference annotation'}"
    )

print("\nDistinct GENCODE gene names:", len(reference_genes))

7.21: Sirt2
7.395: Rps12-ps4
7.422: Mag
7.9: Sirt2

Distinct GENCODE gene names: 3


## 11. Gene-Level Read Counting

Prepare gene-level raw counts for differential expression analysis with DESeq2.

Reads are counted against the exonic regions of genes defined in the final PsiCLASS annotation.

The counting procedure must use raw sequencing counts rather than FPKM or normalized expression values.

The resulting matrix contains genes as rows and RNA-seq samples as columns.

In [43]:
# Prepare name-sorted BAM files for DESeq2
SORTED_DIR.mkdir(parents=True, exist_ok=True)
SORTED_BAMS = []

for sample in SAMPLES:
    input_bam, _ = star_files(sample)
    output_bam = SORTED_DIR / f"{sample}.so.bam"

    if not input_bam.is_file():
        raise FileNotFoundError(f"Missing input BAM: {input_bam}")

    # Reuse existing sorted BAM files
    if output_bam.is_file():
        subprocess.run(
            ["samtools", "quickcheck", "-v", str(output_bam)],
            check=True
        )
        print(f"[OK] Existing sorted BAM: {sample}")

    else:
        print(f"Sorting BAM: {sample}", flush=True)

        subprocess.run([
            "samtools", "sort",
            "-n",
            "-@", str(THREADS),
            "-O", "BAM",
            "-o", str(output_bam),
            str(input_bam)
        ], check=True)

        subprocess.run(
            ["samtools", "quickcheck", "-v", str(output_bam)],
            check=True
        )

        print(f"[OK] Sorted: {sample}")

    SORTED_BAMS.append(output_bam)

print("\nAll six name-sorted BAM files are available.")

[OK] Existing sorted BAM: 1mo_Rep1
[OK] Existing sorted BAM: 1mo_Rep2
[OK] Existing sorted BAM: 1mo_Rep3
[OK] Existing sorted BAM: 4mo_Rep1
[OK] Existing sorted BAM: 4mo_Rep2
[OK] Existing sorted BAM: 4mo_Rep3

All six name-sorted BAM files are available.


In [44]:
# Verify name-sorted BAM files
for sample, bam in zip(SAMPLES, SORTED_BAMS):
    if not bam.is_file():
        raise FileNotFoundError(bam)

    # Check BAM integrity
    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True
    )

    # Verify sorting order
    result = subprocess.run(
        ["samtools", "view", "-H", str(bam)],
        capture_output=True,
        text=True,
        check=True
    )

    if "SO:queryname" not in result.stdout:
        raise ValueError(f"Unexpected BAM sorting order: {sample}")

    print(f"[OK] {sample}: queryname sorted")

print("\nAll six BAM files are ready for gene counting.")

[OK] 1mo_Rep1: queryname sorted
[OK] 1mo_Rep2: queryname sorted
[OK] 1mo_Rep3: queryname sorted
[OK] 4mo_Rep1: queryname sorted
[OK] 4mo_Rep2: queryname sorted
[OK] 4mo_Rep3: queryname sorted

All six BAM files are ready for gene counting.


In [45]:
# Generate gene-level raw counts using R
COUNTS_FILE = (DESEQ2_RESULTS_DIR / "raw_counts.1mo_4mo.psiclass.csv")
RSCRIPT = shutil.which("Rscript")

# Verify inputs
if RSCRIPT is None:
    raise FileNotFoundError("Rscript not found.")

if not FINAL_GTF.is_file():
    raise FileNotFoundError(FINAL_GTF)

for bam in SORTED_BAMS:
    if not bam.is_file():
        raise FileNotFoundError(bam)

DESEQ2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Pass file paths and sample names to R
r_env = os.environ.copy()
r_env["PROJECT_GTF"] = str(FINAL_GTF)
r_env["PROJECT_BAMS"] = ";".join(map(str, SORTED_BAMS))
r_env["PROJECT_SAMPLES"] = ";".join(SAMPLES)
r_env["PROJECT_COUNTS"] = str(COUNTS_FILE)

# R code for gene-level counting
r_code = r'''
suppressPackageStartupMessages({
    library(rtracklayer)
    library(GenomicAlignments)
    library(Rsamtools)
    library(SummarizedExperiment)
})

# Read project inputs
gtf_file <- Sys.getenv("PROJECT_GTF")
output_file <- Sys.getenv("PROJECT_COUNTS")

bam_files <- strsplit(
    Sys.getenv("PROJECT_BAMS"), ";", fixed = TRUE
)[[1]]

sample_ids <- strsplit(
    Sys.getenv("PROJECT_SAMPLES"), ";", fixed = TRUE
)[[1]]

# Import PsiCLASS annotation
gtf <- rtracklayer::import(gtf_file, format = "gtf")
exons <- gtf[gtf$type == "exon"]

if (length(exons) == 0) {
    stop("No exons found in the annotation.")
}

# Group exons by gene
gene_ids <- as.character(exons$gene_id)

if (anyNA(gene_ids) || any(gene_ids == "")) {
    stop("Missing gene IDs in the annotation.")
}

exons_by_gene <- split(exons, gene_ids)

cat("Genes in annotation:", length(exons_by_gene), "\n")

# Open name-sorted BAM files
bams <- Rsamtools::BamFileList(
    bam_files,
    yieldSize = 100000
)

# Count reads overlapping genes
se <- GenomicAlignments::summarizeOverlaps(
    features = exons_by_gene,
    reads = bams,
    mode = "Union",
    singleEnd = TRUE,
    ignore.strand = TRUE,
    fragments = FALSE
)

# Extract raw counts
count_matrix <- SummarizedExperiment::assay(se)
colnames(count_matrix) <- sample_ids

# Validate count matrix
stopifnot(ncol(count_matrix) == length(sample_ids))

if (anyNA(count_matrix) || any(count_matrix < 0)) {
    stop("Invalid read counts.")
}

if (sum(count_matrix) == 0) {
    stop("All counts are zero. Check BAM and GTF coordinates.")
}

# Export raw counts
write.csv(
    count_matrix,
    file = output_file,
    row.names = TRUE
)

cat("\nGene-level counting completed!\n")
cat("Genes:", nrow(count_matrix), "\n")
cat("Samples:", ncol(count_matrix), "\n")
cat("Total assigned reads:", sum(count_matrix), "\n")
'''

# Execute R only if the count matrix is absent
if COUNTS_FILE.exists():
    print("[OK] Existing count matrix:", COUNTS_FILE)

else:
    print("Starting gene-level read counting...", flush=True)

    result = subprocess.run(
        [RSCRIPT, "--vanilla", "-e", r_code],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Gene-level read counting failed.")

# Inspect the resulting count matrix
if not COUNTS_FILE.is_file():
    raise FileNotFoundError(COUNTS_FILE)

counts = pd.read_csv(COUNTS_FILE, index_col=0)

if list(counts.columns) != SAMPLES:
    raise ValueError("Count matrix sample names do not match.")

print("\nRaw count matrix:")
print("Genes:", counts.shape[0])
print("Samples:", counts.shape[1])

display(counts.head())

[OK] Existing count matrix: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/DESeq2/Results/raw_counts.1mo_4mo.psiclass.csv

Raw count matrix:
Genes: 145
Samples: 6


,1mo_Rep1,1mo_Rep2,1mo_Rep3,4mo_Rep1,4mo_Rep2,4mo_Rep3
7.106,5,11,20,9,16,9
7.120,110,150,150,149,138,149
7.130,755,937,914,859,847,1003
7.150,932,1193,1100,867,1031,1162
7.160,399,515,512,599,563,648


In [46]:
# Prepare name-sorted BAMs from corrected STAR alignments
CORRECTED_SORTED_DIR = RESULTS_DIR / "Sorted_corrected"
CORRECTED_SORTED_DIR.mkdir(parents=True, exist_ok=True)

CORRECTED_SORTED_BAMS = []

for sample in SAMPLES:
    input_bam = (
        CORRECTED_STAR_DIR
        / sample
        / "Aligned.sortedByCoord.out.bam"
    )

    output_bam = (
        CORRECTED_SORTED_DIR / f"{sample}.so.bam"
    )

    if not input_bam.is_file():
        raise FileNotFoundError(input_bam)

    if output_bam.is_file():
        subprocess.run(
            ["samtools", "quickcheck", "-v", str(output_bam)],
            check=True
        )
        print(f"[OK] Existing sorted BAM: {sample}")

    else:
        print(f"Sorting BAM: {sample}", flush=True)

        subprocess.run([
            "samtools", "sort",
            "-n",
            "-@", str(THREADS),
            "-O", "BAM",
            "-o", str(output_bam),
            str(input_bam)
        ], check=True)

        subprocess.run(
            ["samtools", "quickcheck", "-v", str(output_bam)],
            check=True
        )

        print(f"[OK] Sorted: {sample}")

    CORRECTED_SORTED_BAMS.append(output_bam)

print("\nAll six corrected name-sorted BAM files are available.")

[OK] Existing sorted BAM: 1mo_Rep1
[OK] Existing sorted BAM: 1mo_Rep2
[OK] Existing sorted BAM: 1mo_Rep3
[OK] Existing sorted BAM: 4mo_Rep1
[OK] Existing sorted BAM: 4mo_Rep2
[OK] Existing sorted BAM: 4mo_Rep3

All six corrected name-sorted BAM files are available.


In [47]:
# Verify availability of the txdbmaker R package
result = subprocess.run(
    [
        str(Path.home() / "miniconda3/envs/transcriptomics/bin/Rscript"),
        "--vanilla",
        "-e",
        'cat("txdbmaker installed:", requireNamespace("txdbmaker", quietly=TRUE), "\\n")'
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

txdbmaker installed: TRUE 



In [48]:
# Generate corrected gene-level counts using the course workflow
CORRECTED_DESEQ2_DIR = RESULTS_DIR / "DESeq2_corrected"
CORRECTED_DESEQ2_RESULTS_DIR = CORRECTED_DESEQ2_DIR / "Results"

CORRECTED_COUNTS_FILE = (
    CORRECTED_DESEQ2_RESULTS_DIR
    / "raw_counts.1mo_4mo.psiclass.csv"
)

CORRECTED_DESEQ2_RESULTS_DIR.mkdir(
    parents=True, exist_ok=True
)

# Verify inputs
if not CORRECTED_FINAL_GTF.is_file():
    raise FileNotFoundError(CORRECTED_FINAL_GTF)

for bam in CORRECTED_SORTED_BAMS:
    if not bam.is_file():
        raise FileNotFoundError(bam)

    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True
    )

# Pass inputs to R
r_env = os.environ.copy()
r_env["PROJECT_GTF"] = str(CORRECTED_FINAL_GTF)
r_env["PROJECT_BAMS"] = ";".join(map(str, CORRECTED_SORTED_BAMS))
r_env["PROJECT_SAMPLES"] = ";".join(SAMPLES)
r_env["PROJECT_COUNTS"] = str(CORRECTED_COUNTS_FILE)

r_code = r'''
suppressPackageStartupMessages({
    library(GenomicFeatures)
    library(GenomicAlignments)
    library(Rsamtools)
    library(SummarizedExperiment)
})

# Read inputs
gtf_file <- Sys.getenv("PROJECT_GTF")
output_file <- Sys.getenv("PROJECT_COUNTS")

bam_files <- strsplit(
    Sys.getenv("PROJECT_BAMS"), ";", fixed = TRUE
)[[1]]

sample_ids <- strsplit(
    Sys.getenv("PROJECT_SAMPLES"), ";", fixed = TRUE
)[[1]]

# Build transcript database from PsiCLASS annotation
hse <- txdbmaker::makeTxDbFromGFF(
    gtf_file,
    format = "gtf"
)

# Group exons by gene
exonsByGene <- exonsBy(hse, by = "gene")

cat("Genes in TxDb:", length(exonsByGene), "\n")

# Open name-sorted BAM files
bamLst <- BamFileList(
    bam_files,
    yieldSize = 100000
)

# Count reads using the original course parameters
se <- summarizeOverlaps(
    features = exonsByGene,
    reads = bamLst,
    mode = "Union",
    singleEnd = FALSE,
    ignore.strand = TRUE,
    fragments = TRUE
)

count_matrix <- assay(se)
colnames(count_matrix) <- sample_ids

# Validate results
if (anyNA(count_matrix) || any(count_matrix < 0)) {
    stop("Invalid read counts.")
}

if (sum(count_matrix) == 0) {
    stop("No reads assigned to genes.")
}

# Export raw counts
write.csv(
    count_matrix,
    output_file,
    row.names = TRUE
)

cat("\nCorrected gene counting completed!\n")
cat("Loci:", nrow(count_matrix), "\n")
cat("Samples:", ncol(count_matrix), "\n")
cat("Total assigned counts:", sum(count_matrix), "\n")
'''

# Preserve previous results
if CORRECTED_COUNTS_FILE.is_file() and (
    CORRECTED_COUNTS_FILE.stat().st_size > 0
):
    print("[OK] Corrected count matrix already exists.")

elif CORRECTED_COUNTS_FILE.exists():
    raise FileExistsError(
        f"Empty count matrix: {CORRECTED_COUNTS_FILE}"
    )

else:
    print("Running corrected gene-level counting...", flush=True)

    result = subprocess.run(
        [
            str(Path.home() / "miniconda3/envs/transcriptomics/bin/Rscript"),
            "--vanilla",
            "-e",
            r_code
        ],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Corrected gene counting failed.")

    if result.stderr:
        print(result.stderr)

# Inspect count matrix
corrected_counts = pd.read_csv(
    CORRECTED_COUNTS_FILE,
    index_col=0,
    dtype=str
)

print("\nCorrected count matrix:")
print("Loci:", corrected_counts.shape[0])
print("Samples:", corrected_counts.shape[1])

display(corrected_counts.head())

[OK] Corrected count matrix already exists.

Corrected count matrix:
Loci: 142
Samples: 6


,1mo_Rep1,1mo_Rep2,1mo_Rep3,4mo_Rep1,4mo_Rep2,4mo_Rep3
7.100,189,230,255,162,260,224
7.103,10,13,13,6,9,14
7.104,11,6,8,9,12,7
7.105,98,97,106,83,103,106
7.107,44,66,73,95,62,88


## 12. Differential Gene Expression — DESeq2

Differential gene expression analysis was performed using DESeq2 to compare hippocampal samples from 1-month-old (1mo) and 4-month-old (4mo) mice, with three biological replicates per condition.

Gene-level raw counts were obtained using `GenomicAlignments::summarizeOverlaps` and the strand-annotated PsiCLASS consensus assembly.

**Methods:**
1. Prepare sample metadata and verify sample ordering.
2. Import raw gene counts while preserving PsiCLASS gene identifiers.
3. Estimate size factors and dispersions using DESeq2.
4. Perform differential expression analysis with `design = ~ condition`.
5. Extract results for 4mo versus 1mo, using 1mo as the reference.
6. Export complete results, normalized counts, and significant genes.
7. Summarize differential expression results.

Statistical significance was assessed using p-values and Benjamini–Hochberg adjusted p-values (FDR), with a threshold of 0.05.

**Input:** Gene-level raw counts and sample metadata.

**Output:** DESeq2 results, normalized counts, and significant gene tables.

In [49]:
# Prepare DESeq2 sample metadata
PHENO_FILE = DESEQ2_DIR / "pheno_1mo_4mo.csv"

# Use original sample metadata
pheno = metadata[["Run", "Age"]].copy()

pheno = pheno.rename(columns={"Run": "sampleID", "Age": "condition"})

pheno["condition"] = pheno["condition"].replace({
    "1_Month": "1mo",
    "4_Month": "4mo"
})

# Match count matrix sample order
pheno = pheno.set_index("sampleID").loc[SAMPLES].reset_index()

if list(pheno["sampleID"]) != list(counts.columns):
    raise ValueError("Sample names do not match the count matrix.")

expected_groups = {"1mo": 3, "4mo": 3}

if pheno["condition"].value_counts().to_dict() != expected_groups:
    raise ValueError("Unexpected experimental group sizes.")

DESEQ2_DIR.mkdir(parents=True, exist_ok=True)

# Preserve existing metadata
if PHENO_FILE.exists():
    existing_pheno = pd.read_csv(PHENO_FILE)
    if not existing_pheno.equals(pheno):
        raise FileExistsError(
            "Existing DESeq2 metadata differs from current metadata."
        )
    print("[OK] Existing DESeq2 metadata verified.")

else:
    pheno.to_csv(PHENO_FILE, index=False)
    print("[OK] DESeq2 metadata created.")

display(pheno)

[OK] Existing DESeq2 metadata verified.


,sampleID,condition
0,1mo_Rep1,1mo
1,1mo_Rep2,1mo
2,1mo_Rep3,1mo
3,4mo_Rep1,4mo
4,4mo_Rep2,4mo
5,4mo_Rep3,4mo


In [50]:
# Run DESeq2 differential expression analysis
RESULTS_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.psiclass.csv"
PVAL_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.pval05.psiclass.csv"
QVAL_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.qval05.psiclass.csv"
NORMALIZED_FILE = DESEQ2_RESULTS_DIR / "normalized_counts.1mo_4mo.psiclass.csv"

output_files = [RESULTS_FILE, PVAL_FILE, QVAL_FILE, NORMALIZED_FILE]

RSCRIPT = str(Path.home() / "miniconda3/envs/transcriptomics/bin/Rscript")

# Verify inputs
for path in [COUNTS_FILE, PHENO_FILE, Path(RSCRIPT)]:
    if not path.is_file():
        raise FileNotFoundError(path)

DESEQ2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Pass file paths to R
r_env = os.environ.copy()
r_env["PROJECT_COUNTS"] = str(COUNTS_FILE)
r_env["PROJECT_PHENO"] = str(PHENO_FILE)
r_env["PROJECT_RESULTS_DIR"] = str(DESEQ2_RESULTS_DIR)

# DESeq2 analysis in R
r_code = r'''
suppressPackageStartupMessages(library(DESeq2))

# Import raw counts preserving PsiCLASS gene IDs
counts_df <- read.csv(
    Sys.getenv("PROJECT_COUNTS"),
    check.names = FALSE,
    colClasses = "character"
)

gene_ids <- counts_df[[1]]

if (anyNA(gene_ids) || any(gene_ids == "") ||
    anyDuplicated(gene_ids)) {
    stop("Missing or duplicated gene IDs.")
}

counts <- as.matrix(counts_df[, -1, drop = FALSE])
rownames(counts) <- gene_ids
storage.mode(counts) <- "numeric"

if (anyNA(counts) || any(counts < 0) ||
    any(counts != round(counts))) {
    stop("Counts must be non-negative integers.")
}

storage.mode(counts) <- "integer"

# Import sample metadata
metadata <- read.csv(
    Sys.getenv("PROJECT_PHENO"),
    row.names = 1
)

if (!setequal(colnames(counts), rownames(metadata))) {
    stop("Sample names do not match.")
}

metadata <- metadata[colnames(counts), , drop = FALSE]

# Set reference condition
metadata$condition <- factor(
    metadata$condition,
    levels = c("1mo", "4mo")
)

if (anyNA(metadata$condition)) {
    stop("Invalid experimental conditions.")
}

# Create DESeq2 dataset
dds <- DESeqDataSetFromMatrix(
    countData = counts,
    colData = metadata,
    design = ~ condition
)

# Estimate size factors, dispersions, and test
dds <- DESeq(dds)

# Extract 4mo vs 1mo results
res <- results(
    dds,
    contrast = c("condition", "4mo", "1mo")
)

res_df <- as.data.frame(res)
outdir <- Sys.getenv("PROJECT_RESULTS_DIR")

# Export complete results
write.csv(
    res_df,
    file.path(outdir, "results_1mo_4mo.psiclass.csv")
)

# Export normalized counts
write.csv(
    counts(dds, normalized = TRUE),
    file.path(outdir, "normalized_counts.1mo_4mo.psiclass.csv")
)

# Filter by unadjusted p-value
res_pval <- res_df[
    !is.na(res_df$pvalue) & res_df$pvalue <= 0.05,
    ,
    drop = FALSE
]

write.csv(
    res_pval,
    file.path(outdir, "results_1mo_4mo.pval05.psiclass.csv")
)

# Filter by adjusted p-value (FDR)
res_qval <- res_df[
    !is.na(res_df$padj) & res_df$padj <= 0.05,
    ,
    drop = FALSE
]

write.csv(
    res_qval,
    file.path(outdir, "results_1mo_4mo.qval05.psiclass.csv")
)

# Display summary
cat("\nDESeq2 analysis completed!\n")
cat("Genes:", nrow(res_df), "\n")
cat("Samples:", ncol(counts), "\n")
cat("p-value <= 0.05:", nrow(res_pval), "\n")
cat("padj <= 0.05:", nrow(res_qval), "\n")
'''

# Reuse existing results without rerunning
if all(
    path.is_file() and path.stat().st_size > 0
    for path in output_files
):
    print("[OK] Existing DESeq2 results found.")

elif any(path.exists() for path in output_files):
    raise FileExistsError(
        "Partial DESeq2 results detected. Inspect before rerunning."
    )

else:
    print("Running DESeq2...", flush=True)

    result = subprocess.run(
        [RSCRIPT, "--vanilla", "-e", r_code],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("DESeq2 analysis failed.")

    if result.stderr:
        print(result.stderr)

# Verify output files
for path in output_files:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty result: {path}")

    print(f"[OK] {path.name}")

[OK] Existing DESeq2 results found.
[OK] results_1mo_4mo.psiclass.csv
[OK] results_1mo_4mo.pval05.psiclass.csv
[OK] results_1mo_4mo.qval05.psiclass.csv
[OK] normalized_counts.1mo_4mo.psiclass.csv


In [51]:
# Summarize DESeq2 differential expression results
# Preserve PsiCLASS gene IDs as strings
deseq_results = pd.read_csv(
    RESULTS_FILE,
    index_col=0,
    dtype=str
)

deseq_results.index = deseq_results.index.astype(str)

# Convert statistical columns
for column in [
    "baseMean", "log2FoldChange",
    "lfcSE", "stat", "pvalue", "padj"
]:
    deseq_results[column] = pd.to_numeric(
        deseq_results[column],
        errors="coerce"
    )

# Identify significant genes
pval_sig = deseq_results["pvalue"].le(0.05)
fdr_sig = deseq_results["padj"].le(0.05)

summary = pd.DataFrame({
    "Metric": [
        "Total genes",
        "Genes with p-value <= 0.05",
        "Genes with padj <= 0.05",
        "Upregulated genes (padj <= 0.05)",
        "Downregulated genes (padj <= 0.05)"
    ],
    "Count": [
        len(deseq_results),
        int(pval_sig.sum()),
        int(fdr_sig.sum()),
        int((fdr_sig & (deseq_results["log2FoldChange"] > 0)).sum()),
        int((fdr_sig & (deseq_results["log2FoldChange"] < 0)).sum())
    ]
})

print("DESeq2 results summary:")
display(summary)

# Display significant genes
significant_genes = (
    deseq_results.loc[
        fdr_sig,
        ["baseMean", "log2FoldChange", "pvalue", "padj"]
    ]
    .sort_values("padj")
)

print("\nSignificant genes (padj <= 0.05):")
display(significant_genes)

DESeq2 results summary:


,Metric,Count
0,Total genes,145
1,Genes with p-value <= 0.05,7
2,Genes with padj <= 0.05,4
3,Upregulated genes (padj <= 0.05),0
4,Downregulated genes (padj <= 0.05),4



Significant genes (padj <= 0.05):


,baseMean,log2FoldChange,pvalue,padj
7.86,983.157986,-0.865733,1.407494e-16,2.040867e-14
7.80,412.613162,-1.004751,1.636210e-15,1.186252e-13
7.73,3319.065576,-0.680576,5.249751e-14,2.537380e-12
7.501,227.475814,-0.710023,3.916798e-06,1.419839e-04


In [52]:
# Assess reference gene significance and expression fold changes
# Preserve PsiCLASS gene identifiers
results = pd.read_csv(RESULTS_FILE, index_col=0, dtype=str)

results["pvalue"] = pd.to_numeric(
    results["pvalue"], errors="coerce"
)
results["log2FoldChange"] = pd.to_numeric(
    results["log2FoldChange"], errors="coerce"
)

# Significant PsiCLASS loci (unadjusted p-value)
significant_loci = results.loc[results["pvalue"] <= 0.05]

# Reference gene names associated with significant loci
reference_genes = set()

for gene_id in significant_loci.index:
    reference_genes.update(
        gene_names.get(str(gene_id), set()) & reference_names
    )

# Loci with absolute log2 fold change >= 2
large_fc = results["log2FoldChange"].abs() >= 2

print("Distinct reference GENCODE gene names:",
      len(reference_genes))
print("Reference genes:", ", ".join(sorted(reference_genes)))

print("\nLoci with |log2FoldChange| >= 2:",
      int(large_fc.sum()))

NameError: name 'gene_names' is not defined

In [ ]:
# Estimate average sequencing depth across Mag exons
# Extract Mag exons from the reference annotation
mag_exons = []

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9 or fields[2] != "exon":
            continue

        if re.search(r'gene_name "Mag"', fields[8]):
            mag_exons.append((
                fields[0],
                int(fields[3]) - 1,
                int(fields[4])
            ))

if not mag_exons:
    raise ValueError("Mag exons not found in GENCODE.")

# Merge overlapping exons to avoid counting bases twice
mag_exons.sort()
merged = []

for chrom, start, end in mag_exons:
    if merged and merged[-1][0] == chrom and start <= merged[-1][2]:
        merged[-1] = (
            chrom, merged[-1][1], max(merged[-1][2], end)
        )
    else:
        merged.append((chrom, start, end))

exonic_length = sum(end - start for _, start, end in merged)

# Calculate coverage using the original STAR BAM files
with tempfile.TemporaryDirectory() as tmp:
    bed = Path(tmp) / "Mag_exons.bed"

    bed.write_text(
        "".join(
            f"{chrom}\t{start}\t{end}\n"
            for chrom, start, end in merged
        )
    )

    bam_files = [str(star_files(s)[0]) for s in SAMPLES]

    result = subprocess.run(
        ["samtools", "depth", "-aa", "-b", str(bed), *bam_files],
        capture_output=True,
        text=True,
        check=True
    )

depth_totals = [0] * len(SAMPLES)
positions = 0

for line in result.stdout.splitlines():
    fields = line.split("\t")
    positions += 1

    for i, depth in enumerate(fields[2:]):
        depth_totals[i] += int(depth)

if positions != exonic_length:
    raise ValueError(
        f"Expected {exonic_length} exonic bases, found {positions}."
    )

coverage = pd.DataFrame({
    "Sample": SAMPLES,
    "Average depth": [
        total / exonic_length for total in depth_totals
    ]
})

print("Mag exonic length:", exonic_length, "bp")
display(coverage)

print(
    "\nMean coverage across six samples:",
    round(coverage["Average depth"].mean())
)

Mag exonic length: 4072 bp


,Sample,Average depth
0,1mo_Rep1,59.975442
1,1mo_Rep2,77.479371
2,1mo_Rep3,66.020874
3,4mo_Rep1,32.048625
4,4mo_Rep2,29.870334
5,4mo_Rep3,37.270629



Mean coverage across six samples: 50


### Annotation Quality Assessment

Inspect gene identifiers, reference annotations, and genomic coordinates to assess potential ambiguities in the PsiCLASS assembly.

In [ ]:
# Map significant PsiCLASS features to GENCODE gene names
gene_names = defaultdict(set)

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attributes = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        gene_id = attributes.get("gene_id")
        gene_name = attributes.get("gene_name")

        if gene_id and gene_name:
            gene_names[gene_id].add(gene_name)

# Preserve all gene names associated with each PsiCLASS ID
annotated_significant = significant_genes.copy()

annotated_significant.insert(
    0,
    "gene_name",
    [
        ", ".join(sorted(gene_names.get(gene_id, []))) or "Unannotated"
        for gene_id in annotated_significant.index
    ]
)

print("Significant PsiCLASS features with GENCODE annotations:")
display(annotated_significant)

Significant PsiCLASS features with GENCODE annotations:


,gene_name,baseMean,log2FoldChange,pvalue,padj
7.86,Sirt2,983.157986,-0.865733,1.407494e-16,2.040867e-14
7.80,Sirt2,412.613162,-1.004751,1.636210e-15,1.186252e-13
7.73,"Cox6b1, Mag",3319.065576,-0.680576,5.249751e-14,2.537380e-12
7.501,Sirt2,227.475814,-0.710023,3.916798e-06,1.419839e-04


In [ ]:
# Check PsiCLASS gene IDs and GENCODE annotations
print("Significant PsiCLASS gene IDs:")

for gene_id in significant_genes.index:
    names = sorted(gene_names.get(str(gene_id), set()))

    print(
        f"gene_id={gene_id!r} | "
        f"gene_names={', '.join(names) if names else 'Not found'}"
    )

print("\nAnnotation summary:")

annotation_summary = pd.DataFrame({
    "gene_id": annotated_significant.index,
    "gene_names": annotated_significant["gene_name"].values,
    "n_gene_names": [
        len(gene_names.get(str(gene_id), set()))
        for gene_id in annotated_significant.index
    ]
})

display(annotation_summary)

Significant PsiCLASS gene IDs:
gene_id='7.86' | gene_names=Sirt2
gene_id='7.80' | gene_names=Sirt2
gene_id='7.73' | gene_names=Cox6b1, Mag
gene_id='7.501' | gene_names=Sirt2

Annotation summary:


,gene_id,gene_names,n_gene_names
0,7.86,Sirt2,1
1,7.80,Sirt2,1
2,7.73,"Cox6b1, Mag",2
3,7.501,Sirt2,1


### Annotation Quality Assessment

Inspect the GENCODE gene names and genomic coordinates associated with significant PsiCLASS features.

These checks help identify ambiguous gene assignments and assess whether transcript structures were already present in the original PsiCLASS assembly.

In [ ]:
# Inspect genomic coordinates of significant PsiCLASS genes
target_ids = set(significant_genes.index.astype(str))
gene_coordinates = []

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attributes = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))

        gene_id = attributes.get("gene_id")

        if gene_id in target_ids:
            gene_coordinates.append({
                "gene_id": gene_id,
                "gene_name": attributes.get("gene_name"),
                "chromosome": fields[0],
                "start": int(fields[3]),
                "end": int(fields[4]),
                "strand": fields[6],
                "transcript_id": attributes.get("transcript_id")
            })

coordinates_df = pd.DataFrame(gene_coordinates)

display(coordinates_df.sort_values(["gene_name", "start", "end"]))

,gene_id,gene_name,chromosome,start,end,strand,transcript_id
1,7.73,Cox6b1,7,1916391,1925577,-,7.73.1
0,7.73,Cox6b1,7,1923925,1925577,-,7.73.0
4,7.73,Mag,7,2198601,2213799,-,7.73.4
3,7.73,Mag,7,2198601,2214329,-,7.73.3
5,7.73,Mag,7,2198601,2214329,-,7.73.5
2,7.73,Mag,7,2198601,2237927,-,7.73.2
6,7.80,Sirt2,7,66191,78414,+,7.80.0
8,7.80,Sirt2,7,66191,78414,+,7.80.2
7,7.80,Sirt2,7,66191,1704069,+,7.80.1
10,7.86,Sirt2,7,82191,87679,+,7.86.1


In [ ]:
# Check annotation consistency for significant PsiCLASS genes
target_ids = set(significant_genes.index.astype(str))
annotations = defaultdict(lambda: defaultdict(set))

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        attrs = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        gene_id = attrs.get("gene_id")

        if gene_id in target_ids:
            annotations[gene_id]["gene_names"].add(
                attrs.get("gene_name", "NA")
            )
            annotations[gene_id]["transcripts"].add(
                attrs.get("transcript_id", "NA")
            )
            annotations[gene_id]["strands"].add(fields[6])

rows = []

for gene_id in sorted(target_ids):
    info = annotations[gene_id]

    rows.append({
        "gene_id": gene_id,
        "gene_names": ", ".join(sorted(info["gene_names"])),
        "n_transcripts": len(info["transcripts"]),
        "strands": ", ".join(sorted(info["strands"])),
        "n_gene_names": len(info["gene_names"])
    })

annotation_check = pd.DataFrame(rows)
display(annotation_check)

,gene_id,gene_names,n_transcripts,strands,n_gene_names
0,7.501,Sirt2,1,+,1
1,7.73,"Cox6b1, Mag",6,-,2
2,7.80,Sirt2,3,+,1
3,7.86,Sirt2,2,+,1


In [ ]:
# Compare original and annotated PsiCLASS transcripts
target_ids = set(significant_genes.index.astype(str))

def extract_transcripts(gtf_file, source):
    records = []

    with open(gtf_file) as gtf:
        for line in gtf:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9 or fields[2] != "transcript":
                continue

            attrs = dict(
                re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
            )

            gene_id = attrs.get("gene_id")

            if gene_id not in target_ids:
                continue

            records.append({
                "source": source,
                "gene_id": gene_id,
                "transcript_id": attrs.get("transcript_id"),
                "gene_name": attrs.get("gene_name"),
                "chromosome": fields[0],
                "start": int(fields[3]),
                "end": int(fields[4]),
                "strand": fields[6]
            })

    return pd.DataFrame(records)

original = extract_transcripts(PSICLASS_VOTE, "Original PsiCLASS")
annotated = extract_transcripts(FINAL_GTF, "Annotated PsiCLASS")
comparison = pd.concat([original, annotated],ignore_index=True)

display(comparison.sort_values(["gene_id", "transcript_id", "source"]))

,source,gene_id,transcript_id,gene_name,chromosome,start,end,strand
23,Annotated PsiCLASS,7.501,7.501.0,Sirt2,7,87743,88085,+
11,Original PsiCLASS,7.501,7.501.0,None,7,87743,88085,.
12,Annotated PsiCLASS,7.73,7.73.0,Cox6b1,7,1923925,1925577,-
0,Original PsiCLASS,7.73,7.73.0,None,7,1923925,1925577,-
13,Annotated PsiCLASS,7.73,7.73.1,Cox6b1,7,1916391,1925577,-
1,Original PsiCLASS,7.73,7.73.1,None,7,1916391,1925577,-
14,Annotated PsiCLASS,7.73,7.73.2,Mag,7,2198601,2237927,-
2,Original PsiCLASS,7.73,7.73.2,None,7,2198601,2237927,-
15,Annotated PsiCLASS,7.73,7.73.3,Mag,7,2198601,2214329,-
3,Original PsiCLASS,7.73,7.73.3,None,7,2198601,2214329,-



### Interpretation and limitations

DESeq2 identified four significant PsiCLASS gene-level features (adjusted p-value ≤ 0.05), all showing lower expression in 4mo compared with 1mo.

Annotation inspection revealed that multiple PsiCLASS gene identifiers were associated with Sirt2, while one identifier contained transcripts annotated as both Cox6b1 and Mag. These assignments were already present in the original PsiCLASS assembly structure.

Therefore, the results are reported at the PsiCLASS feature level rather than interpreted as four independent reference genes. Reference-guided gene counting would be useful for additional validation.

In [ ]:

# Run DESeq2 using corrected PsiCLASS gene counts

CORRECTED_RESULTS_FILE = (
    CORRECTED_DESEQ2_RESULTS_DIR / "results_1mo_4mo.psiclass.csv"
)

CORRECTED_PVAL_FILE = (
    CORRECTED_DESEQ2_RESULTS_DIR / "results_1mo_4mo.pval05.psiclass.csv"
)

CORRECTED_QVAL_FILE = (
    CORRECTED_DESEQ2_RESULTS_DIR / "results_1mo_4mo.qval05.psiclass.csv"
)

CORRECTED_NORMALIZED_FILE = (
    CORRECTED_DESEQ2_RESULTS_DIR / "normalized_counts.1mo_4mo.psiclass.csv"
)

corrected_outputs = [
    CORRECTED_RESULTS_FILE,
    CORRECTED_PVAL_FILE,
    CORRECTED_QVAL_FILE,
    CORRECTED_NORMALIZED_FILE
]

# Reuse the validated sample metadata
for path in [CORRECTED_COUNTS_FILE, PHENO_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

r_env = os.environ.copy()
r_env["PROJECT_COUNTS"] = str(CORRECTED_COUNTS_FILE)
r_env["PROJECT_PHENO"] = str(PHENO_FILE)
r_env["PROJECT_RESULTS_DIR"] = str(CORRECTED_DESEQ2_RESULTS_DIR)

r_code = r'''
suppressPackageStartupMessages(library(DESeq2))

# Preserve PsiCLASS gene identifiers
counts_df <- read.csv(
    Sys.getenv("PROJECT_COUNTS"),
    check.names = FALSE,
    colClasses = "character"
)

gene_ids <- counts_df[[1]]

if (anyNA(gene_ids) || any(gene_ids == "") ||
    anyDuplicated(gene_ids)) {
    stop("Invalid PsiCLASS gene IDs.")
}

counts <- as.matrix(counts_df[, -1, drop = FALSE])
rownames(counts) <- gene_ids
storage.mode(counts) <- "numeric"

if (anyNA(counts) || any(counts < 0) ||
    any(counts != round(counts))) {
    stop("Invalid raw counts.")
}

storage.mode(counts) <- "integer"

# Import sample metadata
metadata <- read.csv(
    Sys.getenv("PROJECT_PHENO"),
    row.names = 1
)

if (!setequal(colnames(counts), rownames(metadata))) {
    stop("Sample names do not match.")
}

metadata <- metadata[colnames(counts), , drop = FALSE]

metadata$condition <- factor(
    metadata$condition,
    levels = c("1mo", "4mo")
)

# Fit DESeq2 model
dds <- DESeqDataSetFromMatrix(
    countData = counts,
    colData = metadata,
    design = ~ condition
)

dds <- DESeq(dds)

res <- results(
    dds,
    contrast = c("condition", "4mo", "1mo")
)

res_df <- as.data.frame(res)
outdir <- Sys.getenv("PROJECT_RESULTS_DIR")

# Export complete results
write.csv(
    res_df,
    file.path(outdir, "results_1mo_4mo.psiclass.csv")
)

# Export normalized counts
write.csv(
    counts(dds, normalized = TRUE),
    file.path(outdir, "normalized_counts.1mo_4mo.psiclass.csv")
)

# Filter by p-value
res_pval <- res_df[
    !is.na(res_df$pvalue) & res_df$pvalue <= 0.05,
    ,
    drop = FALSE
]

write.csv(
    res_pval,
    file.path(outdir, "results_1mo_4mo.pval05.psiclass.csv")
)

# Filter by adjusted p-value
res_qval <- res_df[
    !is.na(res_df$padj) & res_df$padj <= 0.05,
    ,
    drop = FALSE
]

write.csv(
    res_qval,
    file.path(outdir, "results_1mo_4mo.qval05.psiclass.csv")
)

cat("\nCorrected DESeq2 analysis completed!\n")
cat("Loci:", nrow(res_df), "\n")
cat("Samples:", ncol(counts), "\n")
cat("p-value <= 0.05:", nrow(res_pval), "\n")
cat("padj <= 0.05:", nrow(res_qval), "\n")
cat(
    "|log2FoldChange| >= 2:",
    sum(abs(res_df$log2FoldChange) >= 2, na.rm = TRUE),
    "\n"
)
'''

# Preserve existing results
if all(
    path.is_file() and path.stat().st_size > 0
    for path in corrected_outputs
):
    print("[OK] Corrected DESeq2 results already exist.")

elif any(path.exists() for path in corrected_outputs):
    raise FileExistsError(
        "Partial corrected DESeq2 results detected."
    )

else:
    print("Running corrected DESeq2...", flush=True)

    result = subprocess.run(
        [
            str(Path.home() / "miniconda3/envs/transcriptomics/bin/Rscript"),
            "--vanilla", "-e", r_code
        ],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Corrected DESeq2 analysis failed.")

    if result.stderr:
        print(result.stderr)

# Verify results
for path in corrected_outputs:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)

    print(f"[OK] {path.name}")


Running corrected DESeq2...

Corrected DESeq2 analysis completed!
Loci: 142 
Samples: 6 
p-value <= 0.05: 6 
padj <= 0.05: 3 
|log2FoldChange| >= 2: 0 

Warning message:
package ‘S4Vectors’ was built under R version 4.5.3 
estimating size factors
estimating dispersions
gene-wise dispersion estimates
mean-dispersion relationship
final dispersion estimates
fitting model and testing

[OK] results_1mo_4mo.psiclass.csv
[OK] results_1mo_4mo.pval05.psiclass.csv
[OK] results_1mo_4mo.qval05.psiclass.csv
[OK] normalized_counts.1mo_4mo.psiclass.csv


In [ ]:
# Assess reference gene significance and normalized read counts
# Load corrected DESeq2 results
corrected_deseq = pd.read_csv(
    CORRECTED_RESULTS_FILE,
    index_col=0,
    dtype=str
)

for column in ["baseMean", "pvalue", "padj", "log2FoldChange"]:
    corrected_deseq[column] = pd.to_numeric(
        corrected_deseq[column],
        errors="coerce"
    )

# Map PsiCLASS loci to GENCODE gene names
corrected_gene_names = defaultdict(set)

with CORRECTED_FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attrs = dict(
            re.findall(r'(\w+)\s+"([^"]+)"', fields[8])
        )

        gene_id = attrs.get("gene_id")
        gene_name = attrs.get("gene_name")

        if gene_id and gene_name and "novel" not in gene_name.lower():
            corrected_gene_names[gene_id].add(gene_name)

# Q18: Distinct reference genes among significant loci
significant_loci = corrected_deseq.loc[
    corrected_deseq["pvalue"].le(0.05)
]

significant_reference_genes = set()

for locus in significant_loci.index:
    significant_reference_genes.update(
        corrected_gene_names.get(str(locus), set())
    )

print("Significant reference GENCODE genes:")
print(", ".join(sorted(significant_reference_genes)))
print("Count:", len(significant_reference_genes))

# Q20: Inspect Mag-associated loci and baseMean
mag_loci = [
    locus for locus, names in corrected_gene_names.items()
    if "Mag" in names
]

mag_results = corrected_deseq.loc[
    corrected_deseq.index.isin(mag_loci),
    ["baseMean", "log2FoldChange", "pvalue", "padj"]
].copy()

mag_results.insert(0, "gene_name", "Mag")

print("\nMag-associated loci:")
display(mag_results)

Significant reference GENCODE genes:
Arhgap33, C230062I16Rik, Mag, Rps12-ps4, Sirt2
Count: 5

Mag-associated loci:


,gene_name,baseMean,log2FoldChange,pvalue,padj
7.422,Mag,2101.912651,-1.144107,1.702659e-28,2.417775e-26


## 13. Comparison of Differential Expression Results

Compare gene-level differential expression results from Cuffdiff2 and DESeq2 for the 4mo versus 1mo comparison.

**Methods:**
1. Load Cuffdiff2 and DESeq2 results.
2. Identify significant PsiCLASS features using adjusted p-values (FDR ≤ 0.05).
3. Compare significant feature identifiers between methods.
4. Examine log2 fold changes and their directions.

Both methods used the same PsiCLASS consensus annotation. Results are interpreted at the PsiCLASS feature level because some assembled gene identifiers do not correspond uniquely to reference genes.

In [ ]:
# Compare Cuffdiff2 and DESeq2 significant features
# Load results, preserving PsiCLASS IDs
cuff = pd.read_csv(GENE_DIFF, sep="\t", dtype=str)
deseq = pd.read_csv(RESULTS_FILE, index_col=0, dtype=str)

# Convert statistical columns
for column in ["q_value", "log2(fold_change)"]:
    cuff[column] = pd.to_numeric(cuff[column], errors="coerce")

for column in ["padj", "log2FoldChange"]:
    deseq[column] = pd.to_numeric(deseq[column], errors="coerce")

# Identify significant features (FDR <= 0.05)
cuff_sig = set(
    cuff.loc[
        cuff["q_value"].le(0.05) & cuff["status"].eq("OK"),
        "gene_id"
    ]
)

deseq_sig = set(deseq.index[deseq["padj"].le(0.05)])

shared = cuff_sig & deseq_sig

# Display summary
comparison_summary = pd.DataFrame({
    "Method": [
        "Cuffdiff2",
        "DESeq2",
        "Shared",
        "Cuffdiff2 only",
        "DESeq2 only"
    ],
    "Significant features": [
        len(cuff_sig),
        len(deseq_sig),
        len(shared),
        len(cuff_sig - deseq_sig),
        len(deseq_sig - cuff_sig)
    ]
})

print("Differential expression comparison:")
display(comparison_summary)

# Compare significant feature IDs
all_significant = sorted(cuff_sig | deseq_sig)

cuff_lookup = cuff.drop_duplicates("gene_id").set_index("gene_id")

comparison_table = pd.DataFrame(index=all_significant)
comparison_table.index.name = "gene_id"

comparison_table["Cuffdiff2_FDR"] = cuff_lookup["q_value"]
comparison_table["DESeq2_FDR"] = deseq["padj"]

comparison_table["Cuffdiff2_log2FC"] = cuff_lookup["log2(fold_change)"]
comparison_table["DESeq2_log2FC"] = deseq["log2FoldChange"]

comparison_table["Cuffdiff2_significant"] = (comparison_table.index.isin(cuff_sig))
comparison_table["DESeq2_significant"] = (comparison_table.index.isin(deseq_sig))

print("\nSignificant PsiCLASS features:")
display(comparison_table)

Differential expression comparison:


,Method,Significant features
0,Cuffdiff2,4
1,DESeq2,4
2,Shared,3
3,Cuffdiff2 only,1
4,DESeq2 only,1



Significant PsiCLASS features:


,Cuffdiff2_FDR,DESeq2_FDR,Cuffdiff2_log2FC,DESeq2_log2FC,Cuffdiff2_significant,DESeq2_significant
gene_id,,,,,,
7.17,0.001813,7.841602e-01,-0.348179,-0.225856,True,False
7.501,0.001813,1.419839e-04,-0.740333,-0.710023,True,True
7.73,0.947021,2.537380e-12,-0.223788,-0.680576,False,True
7.80,0.001813,1.186252e-13,-1.064320,-1.004751,True,True
7.86,0.001813,2.040867e-14,-1.196140,-0.865733,True,True


### Results and Interpretation

Cuffdiff2 and DESeq2 each identified four significant PsiCLASS features (FDR ≤ 0.05), with three features shared between methods: `7.501`, `7.80`, and `7.86`.

Feature `7.17` was significant only in Cuffdiff2, while `7.73` was significant only in DESeq2. All five features showed negative log2 fold changes in both analyses, indicating lower estimated expression at 4 months compared with 1 month.

Although the methods showed agreement in the direction of expression changes, differences in statistical significance were observed.

These results should be interpreted at the PsiCLASS feature level because some assembled gene identifiers map ambiguously to reference genes.

In [ ]:
# Summarize generated analysis files
OUTPUTS = {
    "STAR index": STAR_INDEX_DIR / "Genome",
    "PsiCLASS assembly": PSICLASS_VOTE,
    "Annotated GTF": FINAL_GTF,
    "Cuffdiff2 results": GENE_DIFF,
    "Cuffdiff2 FPKM tracking": CUFFDIFF_DIR / "genes.fpkm_tracking",
    "DESeq2 count matrix": COUNTS_FILE,
    "DESeq2 metadata": PHENO_FILE,
    "DESeq2 results": RESULTS_FILE,
    "DESeq2 normalized counts": NORMALIZED_FILE,
    "DESeq2 p-value <= 0.05": PVAL_FILE,
    "DESeq2 FDR <= 0.05": QVAL_FILE
}

summary = pd.DataFrame([
    {
        "Analysis": name,
        "Status": (
            "Available"
            if path.is_file() and path.stat().st_size > 0
            else "Missing"
        ),
        "Path": str(path)
    }
    for name, path in OUTPUTS.items()
])

print("Final project output summary:")
display(summary)

print(
    f"\nAvailable files: "
    f"{(summary['Status'] == 'Available').sum()}/{len(summary)}")

Final project output summary:


,Analysis,Status,Path
0,STAR index,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
1,PsiCLASS assembly,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
2,Annotated GTF,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
3,Cuffdiff2 results,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
4,Cuffdiff2 FPKM tracking,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
5,DESeq2 count matrix,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
6,DESeq2 metadata,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
7,DESeq2 results,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
8,DESeq2 normalized counts,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...
9,DESeq2 p-value <= 0.05,Available,/mnt/d/Bioinformatics/transcriptomics-data/Mod...



Available files: 11/11
